# Cospas-Sarsat SAR incident parser - Canada-only JSON, 2007-2024 (v7 final older-layout fixes)

This notebook downloads the annual Cospas-Sarsat `C/S R.007` PDFs listed below, extracts Annex C incident records, and writes a single Canada-only JSON file.

**Canada-only filter used:** modern records use `event_id` starting with `CA-`; older Annex-C-only records without Event IDs use Canadian MCC/location/beacon/flag signals (`CMCC`, `Canada`, `Canadian`, `316`, `CAN`).

The parser is designed for recurring Annex C form layouts from both full reports and Annex-C-only PDFs. It explicitly handles the page-break issue where `Reporting MCC` / `Event ID` appears at the bottom of one page while the rest of that record continues on the following page. It treats `Reporting MCC` + `Event ID` as the start of the next record.

Outputs are written to `outputs/`:

- `cospas_sarsat_incident_records_canada_2007_2024.json`
- `cospas_sarsat_incident_records_canada_2007_2024.csv`
- `cospas_sarsat_yearly_profile.json`
- `cospas_sarsat_page_end_event_id_instances.csv`
- `cospas_sarsat_parse_errors.csv`


In [ ]:
# Install system dependency for Poppler's pdftotext.
# In Colab/Linux this preserves the PDF layout much better than generic PDF text extraction.
!sudo apt-get update
!sudo apt-get install -y poppler-utils


In [ ]:
# Python dependencies
# - requests: download PDFs
# - pandas: output CSV / summaries
# - PyMuPDF: fallback PDF text extraction if pdftotext is unavailable
#
# Preferred text extraction is Poppler's pdftotext -layout command because it preserves the Annex C layout well.

import csv
import hashlib
import json
import math
import os
import re
import shutil
import subprocess
import sys
from collections import Counter
from datetime import datetime
from pathlib import Path

import pandas as pd
import requests

try:
    import fitz  # PyMuPDF fallback
except Exception:
    fitz = None

BASE_DIR = Path.cwd()
PDF_DIR = BASE_DIR / "pdfs"
TEXT_DIR = BASE_DIR / "text"
OUTPUT_DIR = BASE_DIR / "outputs"
for d in [PDF_DIR, TEXT_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
PDF_URLS = [
    "https://cospas-sarsat.int/images/stories/SystemDocs/Current/R007-OCT-23-2025.pdf",              # 2024 report
    "https://cospas-sarsat.int/images/stories/SystemDocs/Current/R007-OCT-24-2024.pdf",              # 2023 report
    "https://cospas-sarsat.int/images/stories/SystemDocs/Current/R007-OCT-27-2023.pdf",              # 2022 report
    "https://cospas-sarsat.int/images/stories/SystemDocs/Current/R007-NOV-29-2022.pdf",              # 2021 report
    "https://cospas-sarsat.int/images/stories/SystemDocs/Current/R007-MAR-25-2022.pdf",              # 2020 report
    "https://cospas-sarsat.int/images/stories/SystemDocs/Current/R007-MAR-26-2021.pdf",              # 2019 report
    "https://cospas-sarsat.int/images/stories/SystemDocs/Current/R007-NOV-22-2019.pdf",              # 2018 report
    "https://cospas-sarsat.int/images/stories/SystemDocs/Current/CS-R007-FEB-2019.pdf",              # 2017 report
    "https://cospas-sarsat.int/images/stories/SystemDocs/Current/CS-R007-FEB-2018.pdf",              # 2016 report
    "https://cospas-sarsat.int/images/stories/SystemDocs/Current/CS-R007-DEC-2016.pdf",              # 2015 report
    "https://cospas-sarsat.int/images/stories/SystemDocs/Current/Annex-C-from-R007-DEC-10-2015.pdf", # 2014 Annex C only
    "https://cospas-sarsat.int/images/stories/SystemDocs/Current/R7-OCT30-2014-C.pdf",               # 2013 Annex C / Part C
    "https://cospas-sarsat.int/images/stories/SystemDocs/Current/r7_oct31_2013_c.pdf",               # 2012 Annex C / Part C
    "https://cospas-sarsat.int/images/stories/SystemDocs/Current/r7_oct27_2011_c.pdf",               # 2010 Annex C only
    "https://cospas-sarsat.int/images/stories/SystemDocs/Current/R7OCT28C.10_Part2_AnnexC_smallsize.pdf", # 2009 Annex C only
    "https://cospas-sarsat.int/images/stories/SystemDocs/Current/r7-no25-jan-dec2008-annexc.pdf",    # 2008 Annex C only
    "https://cospas-sarsat.int/images/stories/SystemDocs/Current/R7OCT30-2007-AnnexC.pdf",           # 2007 Annex C only
]

# Expected report years by file name. The PDF title is also parsed and used as a fallback.
URL_YEAR_HINT = {
    "R007-OCT-23-2025": 2024,
    "R007-OCT-24-2024": 2023,
    "R007-OCT-27-2023": 2022,
    "R007-NOV-29-2022": 2021,
    "R007-MAR-25-2022": 2020,
    "R007-MAR-26-2021": 2019,
    "R007-NOV-22-2019": 2018,
    "CS-R007-FEB-2019": 2017,
    "CS-R007-FEB-2018": 2016,
    "CS-R007-DEC-2016": 2015,
    "Annex-C-from-R007-DEC-10-2015": 2014,
    "R7-OCT30-2014-C": 2013,
    "r7_oct31_2013_c": 2012,
    "r7_oct27_2011_c": 2010,
    "R7OCT28C.10_Part2_AnnexC_smallsize": 2009,
    "r7-no25-jan-dec2008-annexc": 2008,
    "R7OCT30-2007-AnnexC": 2007,
}

# Keep running if one historical URL fails, but record the failure in the yearly profile.
STOP_ON_DOWNLOAD_ERROR = False


In [ ]:
def download_pdf(url: str, dest_dir: Path = PDF_DIR, timeout: int = 180) -> Path:
    """Download a PDF. If the URL ends in '.pd' or returns non-PDF content, retry with '.pdf'."""
    candidates = [url]
    if url.endswith(".pd"):
        candidates.append(url + "f")
    elif not url.lower().endswith(".pdf"):
        candidates.append(url + ".pdf")

    headers = {
        "User-Agent": "Mozilla/5.0 (compatible; CospasSarsatDataParser/1.0)",
        "Accept": "application/pdf,*/*;q=0.8",
    }
    last_error = None
    for candidate in candidates:
        filename = candidate.rstrip("/").split("/")[-1]
        if filename.endswith(".pd"):
            filename += "f"
        dest = dest_dir / filename

        if dest.exists() and dest.stat().st_size > 0:
            with open(dest, "rb") as f:
                if f.read(5) == b"%PDF-":
                    return dest
            dest.unlink()

        try:
            print(f"Downloading {candidate}")
            with requests.get(candidate, stream=True, timeout=timeout, headers=headers, allow_redirects=True) as r:
                r.raise_for_status()
                content_type = r.headers.get("content-type", "").lower()
                with open(dest, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1024 * 128):
                        if chunk:
                            f.write(chunk)

            if dest.stat().st_size < 10_000:
                raise ValueError(f"Downloaded file is suspiciously small: {dest.stat().st_size} bytes")
            with open(dest, "rb") as f:
                magic = f.read(5)
            if magic != b"%PDF-":
                raise ValueError(f"Downloaded file is not a PDF; content-type={content_type}, magic={magic!r}")
            return dest
        except Exception as exc:
            last_error = exc
            if dest.exists():
                # remove failed / HTML error downloads
                try:
                    if dest.stat().st_size < 10_000 or open(dest, "rb").read(5) != b"%PDF-":
                        dest.unlink()
                except Exception:
                    pass

    raise RuntimeError(f"Could not download PDF from {url}: {last_error}")


def extract_text_with_pages(pdf_path: Path, text_dir: Path = TEXT_DIR) -> Path:
    """Extract text to a .txt file. Prefer pdftotext -layout; fallback to PyMuPDF if needed."""
    text_path = text_dir / (pdf_path.stem + ".txt")
    if text_path.exists() and text_path.stat().st_size > 0:
        return text_path

    pdftotext = shutil.which("pdftotext")
    if pdftotext:
        cmd = [pdftotext, "-layout", "-enc", "UTF-8", str(pdf_path), str(text_path)]
        subprocess.run(cmd, check=True)
        return text_path

    if fitz is None:
        raise RuntimeError(
            "pdftotext was not found and PyMuPDF is not installed. Install Poppler or `pip install pymupdf`."
        )

    # Fallback: generally works, but pdftotext -layout is better for this form-like PDF layout.
    doc = fitz.open(str(pdf_path))
    with open(text_path, "w", encoding="utf-8") as f:
        for page in doc:
            f.write(page.get_text("text", sort=True))
            f.write("\f")
    return text_path


In [ ]:
def norm_ws(s):
    if s is None:
        return None
    s = re.sub(r"\s+", " ", str(s).replace("￾", "")).strip()
    return s if s else None


def parse_date(raw):
    raw = norm_ws(raw)
    if not raw:
        return None

    for fmt in ["%Y-%m-%d", "%d-%b-%Y", "%d-%b-%y", "%d %b %Y", "%d %b %y", "%d-%B-%Y", "%d-%B-%y"]:
        try:
            return datetime.strptime(raw, fmt).date().isoformat()
        except Exception:
            pass

    # Manual fallback, including Sept/Sep and two-digit years.
    month_map = {
        "jan": 1, "feb": 2, "mar": 3, "apr": 4, "may": 5, "jun": 6,
        "jul": 7, "aug": 8, "sep": 9, "sept": 9, "oct": 10, "nov": 11, "dec": 12,
    }
    m = re.match(r"(\d{1,2})[- ]([A-Za-z]{3,5})[- ](\d{2,4})", raw)
    if m:
        d, mo, y = m.groups()
        yy = int(y)
        if yy < 100:
            yy += 2000
        return f"{yy:04d}-{month_map[mo.lower()]:02d}-{int(d):02d}" if mo.lower() in month_map else None
    return None


def to_float(x):
    x = norm_ws(x)
    if not x:
        return None
    try:
        return float(x.replace(",", ""))
    except Exception:
        return None


def decimal_coord(deg, minutes, hemisphere):
    if deg is None or minutes is None or hemisphere is None:
        return None
    val = float(deg) + float(minutes) / 60.0
    if str(hemisphere).lower() in {"south", "west"}:
        val = -val
    return round(val, 6)


def cleanup_headers(text):
    text = text or ""
    text = re.sub(r"\[\[PDF_PAGE:\d+\]\]", " ", text)
    # General report header/footer cleanup across report numbers and years, with optional Annex C.
    text = re.sub(
        r"C/S Report on System Status and Operations No\.\s*\d+,\s*January\s*-\s*December\s+\d{4}(?:,\s*Annex C)?\s+Page\s+\d+",
        " ",
        text,
    )
    # Older Annex-C-only files can have slightly different running headers.
    text = re.sub(r"Cospas-Sarsat Report on System Status and Operations.*?Page\s+\d+", " ", text, flags=re.I)
    text = re.sub(r"-\s*END OF ANNEX C\s*-", " ", text, flags=re.I)
    return text


def cleanup_record_text(text):
    return norm_ws(cleanup_headers(text))


def infer_report_year(full_text: str, pdf_path: Path) -> int | None:
    # Full reports usually have a title page like: No.40: January - December 2023
    m = re.search(r"No\.\s*\d+\s*:\s*January\s*-\s*December\s+(20\d{2})", full_text[:8000], flags=re.I)
    if m:
        return int(m.group(1))

    # Annex-only documents may say Jan - Dec 2014, Jan. - Dec. 2010, or January - December 2007.
    m = re.search(r"(?:Jan(?:uary)?\.?\s*-\s*Dec(?:ember)?\.?|January\s*-\s*December)\s*(20\d{2})", full_text[:12000], flags=re.I)
    if m:
        return int(m.group(1))

    stem = pdf_path.stem
    for key, year in URL_YEAR_HINT.items():
        if key in stem:
            return year
    return None


def file_md5(path: Path) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


In [ ]:
FIELD_LABELS = [
    "Reporting MCC:", "Reporting RCC:", "MCC:", "RCC:", "Event ID:",
    "Date of Incident", "Date of incident", "Incident Date", "Alert time in GMT", "Alert Time in GMT",
    "Latitude:", "Longitude:", "Type of incident:", "Type of Incident:", "Location of incident:", "Location:",
    "Type of beacon:", "Type of Beacon:", "Beacon Country Code:", "Beacon country code:",
    "Beacon hex:", "Beacon Hex:", "Beacon ID:", "Beacon Hex ID:", "Beacon Frequency:",
    "Were valid beacon registration data available?", "Valid beacon registration data available?",
    "Vehicle type:", "Vehicle Type:", "Vehicle name:", "Vehicle Name:",
    "Vessel/Aircraft/Mobile flag:", "Vessel/Aircraft/Mobile Flag:", "Vehicle/Mobile flag:",
    "Vehicle Call Sign:", "Vehicle call sign:", "Call Sign:",
    "Type of assistance provided by Cospas-Sarsat:", "Type of assistance provided:", "Assistance provided by Cospas-Sarsat:",
    "Number Persons Involved:", "Number of Persons Involved:", "Persons Involved:",
    "Cospas-Sarsat Assisted Saves:", "Cospas Sarsat Assisted Saves:", "Assisted Saves:",
    "Details of Incident:", "Incident Details:", "Details:",
]

COUNTRY_CODE_MAP = {
    "316": "Canada", "366": "United States", "503": "Australia", "512": "New Zealand",
    "227": "France", "228": "France", "234": "United Kingdom", "232": "United Kingdom", "235": "United Kingdom",
    "244": "Netherlands", "247": "Italy", "262": "Germany", "224": "Spain", "257": "Norway", "273": "Russian Federation",
    "412": "China (P.R. of)", "525": "Indonesia", "440": "Japan", "431": "Japan", "470": "United Arab Emirates",
    "345": "Mexico", "725": "Chile", "710": "South Africa", "338": "United States", "367": "United States",
}

KNOWN_FLAG_CODES = {
    "CAN", "USA", "AUS", "NZL", "FRA", "GBR", "GBD", "GBI", "NLD", "ITA", "DEU", "ESP", "NOR",
    "RUS", "CHN", "JPN", "INO", "BRA", "ARG", "CHL", "PER", "MEX", "KOR", "TUR", "GRC", "SWE", "FIN",
    "POL", "DNK", "BEL", "CHE", "ZAF", "IND", "MYS", "SGP", "THA", "VNM", "PAK", "NGA", "TGO",
    "DZA", "MAR", "MTQ", "CYM", "BMU", "PAN", "LBR", "BHS", "VUT", "CYP", "QAT", "SAU", "SRB",
    "TWN", "HKG", "ARE", "TUN", "ISL", "IRL", "PRT", "AUT", "CZE", "URY", "COL", "ECU", "BOL",
}


In [ ]:
# Robust parser helpers for fields that often break when PDF text order is column-major.
ASSISTANCE_VALUES = [
    "data not used in SAR",
    "supporting data",
    "supported data",
    "first alert",
    "only alert",
]

BEACON_TYPES_RE = r"(?:ELT|EPIRB|PLB|SSAS|SGB)"
VEHICLE_TYPE_RE = r"(?:Aircraft\s*-\s*[A-Za-z]+|Vessel\s*-\s*[A-Za-z]+|Aircraft|Helicopter|Airplane|Vessel|Boat|Yacht|Ship|Person|Mobile|Vehicle|Other)"
TERMINAL_VALUES_RE = r"(?:Not Applicable|Not applicable|Unreported|Unavailable|Unknown|None|N/A|Not reported)"

FLAG_COUNTRY_HINTS = {
    "CAN": "Canada", "USA": "United States", "AUS": "Australia", "NZL": "New Zealand",
    "FRA": "France", "GBR": "United Kingdom", "GBD": "United Kingdom", "GBI": "United Kingdom",
    "NLD": "Netherlands", "ITA": "Italy", "DEU": "Germany", "ESP": "Spain", "NOR": "Norway",
    "RUS": "Russian Federation", "CHN": "China (P.R. of)", "JPN": "Japan", "INO": "Indonesia",
    "BRA": "Brazil", "ARG": "Argentina", "CHL": "Chile", "PER": "Peru", "MEX": "Mexico",
    "KOR": "Korea (Rep. of)", "TUR": "Türkiye", "GRC": "Greece", "SWE": "Sweden", "FIN": "Finland",
    "POL": "Poland", "DNK": "Denmark", "BEL": "Belgium", "CHE": "Switzerland", "ZAF": "South Africa",
    "IND": "India", "MYS": "Malaysia", "SGP": "Singapore", "THA": "Thailand", "VNM": "Viet Nam",
    "PAK": "Pakistan", "NGA": "Nigeria", "TGO": "Togo", "DZA": "Algeria", "MAR": "Morocco",
    "MTQ": "Martinique", "CYM": "Cayman Islands", "BMU": "Bermuda", "PAN": "Panama", "LBR": "Liberia",
    "BHS": "Bahamas", "VUT": "Vanuatu", "CYP": "Cyprus", "QAT": "Qatar", "SAU": "Saudi Arabia",
    "SRB": "Serbia", "TWN": "Chinese Taipei", "HKG": "Hong Kong (China)", "ARE": "United Arab Emirates",
    "TUN": "Tunisia", "ISL": "Iceland", "IRL": "Ireland", "PRT": "Portugal", "AUT": "Austria",
    "CZE": "Czechia", "URY": "Uruguay", "COL": "Colombia", "ECU": "Ecuador", "BOL": "Bolivia",
}
COUNTRY_NAME_ALTS = sorted(set(FLAG_COUNTRY_HINTS.values()) | set(COUNTRY_CODE_MAP.values()), key=len, reverse=True)
COUNTRY_NAME_PATTERN = r"(?:" + "|".join(re.escape(x) for x in COUNTRY_NAME_ALTS) + r")"


def strip_page_markers(text: str) -> str:
    return re.sub(r"\[\[PDF_PAGE:\d+\]\]", " ", text or "")


def clean_extracted_value(value):
    value = cleanup_headers(strip_page_markers(value or ""))
    value = re.sub(r"\b(?:degrees|minutes|MHz)\b", " ", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value or None


def collapsed_record(raw: str) -> str:
    """Collapse PDF record text while keeping field labels visible."""
    s = strip_page_markers(raw or "")
    s = cleanup_headers(s)
    s = re.sub(r"\s+", " ", s).strip()
    # Add missing separators around labels that PDF extraction sometimes glues to values.
    for label in sorted(FIELD_LABELS, key=len, reverse=True):
        s = re.sub(rf"(?<!^)(?<!\s)({re.escape(label)})", r" \1", s, flags=re.I)
        s = re.sub(rf"({re.escape(label)})(?!\s)", r"\1 ", s, flags=re.I)
    return re.sub(r"\s+", " ", s).strip()


def find_first_label_pos(s: str, labels: list[str], start: int = 0):
    hits = []
    lower = s.lower()
    for label in labels:
        pos = lower.find(label.lower(), start)
        if pos >= 0:
            hits.append((pos, label))
    return min(hits, default=(None, None), key=lambda x: x[0])


def section_between_any(s: str, start_labels: list[str], stop_labels: list[str]) -> str | None:
    start_pos, start_label = find_first_label_pos(s, start_labels)
    if start_pos is None:
        return None
    stop_pos, _ = find_first_label_pos(s, stop_labels, start_pos + len(start_label))
    end = stop_pos if stop_pos is not None else len(s)
    return s[start_pos:end].strip()


def clean_value_until_label(value: str | None):
    value = clean_extracted_value(value)
    if not value:
        return None
    lower = value.lower()
    cut_positions = [lower.find(label.lower()) for label in FIELD_LABELS if lower.find(label.lower()) >= 0]
    if cut_positions:
        value = value[:min(cut_positions)]
    return clean_extracted_value(value)


def extract_after_label(s: str, labels: list[str], stop_labels: list[str]) -> str | None:
    pos, label = find_first_label_pos(s, labels)
    if pos is None:
        return None
    start = pos + len(label)
    stop_pos, _ = find_first_label_pos(s, stop_labels, start)
    return clean_value_until_label(s[start:stop_pos if stop_pos is not None else len(s)])


def extract_location(raw):
    s = collapsed_record(raw)
    return extract_after_label(
        s,
        ["Location of incident:", "Location:"],
        ["Beacon Country Code:", "Type of beacon:", "Beacon hex:", "Beacon ID:", "Beacon Frequency:",
         "Were valid beacon registration data available?", "Vehicle type:", "Vessel/Aircraft/Mobile flag:",
         "Type of assistance provided by Cospas-Sarsat:", "Number Persons Involved:"]
    )


def extract_beacon(raw):
    s = collapsed_record(raw)
    section = section_between_any(
        s,
        ["Type of beacon:", "Type of Beacon:", "Beacon Country Code:", "Beacon country code:"],
        ["Beacon hex:", "Beacon Hex:", "Beacon ID:", "Beacon Hex ID:", "Beacon Frequency:",
         "Were valid beacon registration data available?", "Vehicle type:", "Vessel/Aircraft/Mobile flag:",
         "Vehicle/Mobile flag:", "Type of assistance provided by Cospas-Sarsat:"]
    ) or ""

    beacon_type = None
    m = re.search(rf"Type of beacon:\s*({BEACON_TYPES_RE})\b", section, flags=re.I)
    if m:
        beacon_type = m.group(1).upper()
    else:
        m = re.search(rf"\b({BEACON_TYPES_RE})\b", section, flags=re.I)
        if m:
            beacon_type = m.group(1).upper()

    country_code = None
    country_name = None
    m = re.search(r"Beacon Country Code:\s*(?:Type of beacon:\s*(?:ELT|EPIRB|PLB|SSAS|SGB)\s*)?(\d{3})\s*([^:]*?)$", section, flags=re.I)
    if not m:
        m = re.search(r"(\d{3})\s*([^:]*?)$", section, flags=re.I)
    if m:
        country_code = clean_value_until_label(m.group(1))
        country_name = clean_value_until_label(m.group(2))

    if country_code is None:
        m = re.search(r"Beacon Country Code:\s*(?:Type of beacon:\s*(?:ELT|EPIRB|PLB|SSAS|SGB)\s*)?(\d{3})\s*([^:]*?)(?:Beacon hex:|Beacon ID:|Beacon Frequency:|Were valid|Vehicle type:|$)", s, flags=re.I)
        if m:
            country_code = clean_value_until_label(m.group(1))
            country_name = clean_value_until_label(m.group(2))

    if country_code and (not country_name or country_name.lower() in {"type", "of", "beacon"}):
        country_name = COUNTRY_CODE_MAP.get(country_code)

    # Older annexes sometimes call this Beacon ID instead of Beacon hex.
    m = re.search(r"(?:Beacon hex|Beacon Hex|Beacon ID|Beacon Hex ID):\s*([A-Fa-f0-9]{5,}|Unreported|Unavailable|Not Applicable|Not applicable|Unknown)", s)
    beacon_hex = None
    if m and re.fullmatch(r"[A-Fa-f0-9]{5,}", m.group(1)):
        beacon_hex = m.group(1).upper()

    m = re.search(r"Beacon Frequency:\s*([0-9]+(?:\.[0-9]+)?)\s*MHz", s, re.I)
    beacon_frequency_mhz = float(m.group(1)) if m else None

    m = re.search(r"(?:Were valid beacon registration data available\?|Valid beacon registration data available\?)\s*(Yes|No)", s, re.I)
    valid_reg = None if not m else (m.group(1).lower() == "yes")

    return beacon_type, beacon_hex, beacon_frequency_mhz, country_code, country_name, valid_reg


def split_name_and_call_sign(rest: str | None):
    rest = clean_value_until_label(rest)
    if not rest:
        return None, None

    m = re.match(rf"^({TERMINAL_VALUES_RE})\s+({TERMINAL_VALUES_RE})$", rest, flags=re.I)
    if m:
        return clean_value_until_label(m.group(1)), clean_value_until_label(m.group(2))

    m = re.match(rf"^(.*?)\s+({TERMINAL_VALUES_RE})$", rest, flags=re.I)
    if m and m.group(1).strip():
        return clean_value_until_label(m.group(1)), clean_value_until_label(m.group(2))

    # Common call sign forms at the end: C-FABC, N123AB, VH-ABC, ZK-ABC, WABC123, digits, etc.
    m = re.match(r"^(.*?)\s+([A-Z0-9]{1,4}-?[A-Z0-9]{2,8}|[A-Z]{1,4}\d{2,6}|\d[-A-Z0-9]{2,10})$", rest)
    if m and m.group(1).strip():
        return clean_value_until_label(m.group(1)), clean_value_until_label(m.group(2))

    parts = rest.rsplit(" ", 1)
    if len(parts) == 2:
        return clean_value_until_label(parts[0]), clean_value_until_label(parts[1])
    return rest, None


def parse_vehicle_fixed_width(raw: str):
    """Fallback for normal pdftotext layout where the vehicle row appears under fixed-width headers."""
    patterns = [
        r"Vehicle type:\s+Vehicle name:\s+(?:Vessel/Aircraft/Mobile flag:|Vehicle/Mobile flag:)\s+(?:Vehicle Call Sign:|Call Sign:)\s*\n(.+?)(?:\n\s*\n|\n\s*(?:Type of assistance|Number Persons|Cospas-Sarsat Assisted Saves|Details of Incident))",
        r"(?:Vessel/Aircraft/Mobile flag:|Vehicle/Mobile flag:)\s+Vehicle type:\s+Vehicle name:\s+(?:Vehicle Call Sign:|Call Sign:)\s*\n(.+?)(?:\n\s*\n|\n\s*(?:Type of assistance|Number Persons|Cospas-Sarsat Assisted Saves|Details of Incident))",
    ]
    for pat in patterns:
        m = re.search(pat, raw, flags=re.S | re.I)
        if not m:
            continue
        lines = [cleanup_headers(strip_page_markers(x)).rstrip() for x in m.group(1).splitlines() if x.strip()]
        if not lines:
            continue
        parts = [p.strip() for p in re.split(r"\s{2,}", lines[0].strip()) if p.strip()]
        if len(parts) >= 5:
            # Normal order: type, name, flag code, flag country, call sign
            if re.match(rf"^{VEHICLE_TYPE_RE}$", parts[0], flags=re.I):
                return parts[0], parts[1], parts[2].upper(), parts[3], " ".join(parts[4:])
            # Reordered order: flag code, flag country, type, name, call sign
            if re.fullmatch(r"[A-Z]{3}", parts[0]):
                return parts[2], parts[3], parts[0].upper(), parts[1], " ".join(parts[4:])
    return None


def parse_vehicle(raw):
    fixed = parse_vehicle_fixed_width(raw)
    if fixed:
        return fixed

    s = collapsed_record(raw)
    section = section_between_any(
        s,
        ["Vehicle type:", "Vehicle Type:", "Vessel/Aircraft/Mobile flag:", "Vehicle/Mobile flag:"],
        ["Type of assistance provided by Cospas-Sarsat:", "Type of assistance provided:",
         "Assistance provided by Cospas-Sarsat:", "Number Persons Involved:", "Number of Persons Involved:",
         "Cospas-Sarsat Assisted Saves:", "Details of Incident:"]
    )
    if not section:
        return None, None, None, None, None

    values = section
    for label in ["Vessel/Aircraft/Mobile flag:", "Vessel/Aircraft/Mobile Flag:", "Vehicle/Mobile flag:",
                  "Vehicle type:", "Vehicle Type:", "Vehicle name:", "Vehicle Name:",
                  "Vehicle Call Sign:", "Vehicle call sign:", "Call Sign:"]:
        values = re.sub(re.escape(label), " ", values, flags=re.I)
    values = clean_value_until_label(values)
    if not values:
        return None, None, None, None, None

    values = re.sub(rf"({COUNTRY_NAME_PATTERN})(?=({VEHICLE_TYPE_RE}))", r"\1 ", values, flags=re.I)
    values = re.sub(r"(?<=[a-z])(?=(Aircraft|Vessel|Person|Mobile|Boat|Helicopter|Airplane|Vehicle)\b)", " ", values)
    values = re.sub(r"\s+", " ", values).strip()

    vehicle_flag_code = vehicle_flag_country = vehicle_type = vehicle_name = vehicle_call_sign = None

    # Reordered/collapsed form: CAN CanadaAircraft -private CESSNA 172F C-GVCO
    m = re.match(rf"^(?P<code>[A-Z]{{3}})\s+(?P<country>{COUNTRY_NAME_PATTERN}|.*?)(?P<vtype>{VEHICLE_TYPE_RE})\s*(?P<rest>.*)$", values, flags=re.I)
    if m:
        vehicle_flag_code = m.group("code").upper()
        vehicle_flag_country = clean_value_until_label(m.group("country")) or FLAG_COUNTRY_HINTS.get(vehicle_flag_code)
        vehicle_type = clean_value_until_label(m.group("vtype"))
        vehicle_name, vehicle_call_sign = split_name_and_call_sign(m.group("rest"))
    else:
        # Normal/collapsed form: Aircraft -private CESSNA 172F CAN Canada C-GVCO
        m = re.match(rf"^(?P<vtype>{VEHICLE_TYPE_RE})\s+(?P<tail>.*)$", values, flags=re.I)
        if m:
            vehicle_type = clean_value_until_label(m.group("vtype"))
            tail = m.group("tail").strip()
            m_code = None
            for candidate in re.finditer(r"\b([A-Z]{3})\b", tail):
                code = candidate.group(1).upper()
                after_candidate = tail[candidate.end():].strip()
                country_hint = FLAG_COUNTRY_HINTS.get(code)
                if country_hint and after_candidate.lower().startswith(country_hint.lower()):
                    m_code = candidate
                elif re.match(rf"{COUNTRY_NAME_PATTERN}\b", after_candidate, flags=re.I):
                    m_code = candidate

            if m_code:
                vehicle_flag_code = m_code.group(1).upper()
                vehicle_name = clean_value_until_label(tail[:m_code.start()])
                after_code = tail[m_code.end():].strip()
                country_hint = FLAG_COUNTRY_HINTS.get(vehicle_flag_code)
                if country_hint and after_code.lower().startswith(country_hint.lower()):
                    vehicle_flag_country = country_hint
                    after_country = after_code[len(country_hint):].strip()
                else:
                    m_country = re.match(rf"(?P<country>{COUNTRY_NAME_PATTERN})\s*(?P<rest>.*)$", after_code, flags=re.I)
                    if m_country:
                        vehicle_flag_country = clean_value_until_label(m_country.group("country"))
                        after_country = m_country.group("rest")
                    else:
                        vehicle_flag_country = country_hint
                        after_country = after_code
                vehicle_call_sign = clean_value_until_label(after_country)
            else:
                vehicle_name, vehicle_call_sign = split_name_and_call_sign(tail)

    if vehicle_flag_code and not vehicle_flag_country:
        vehicle_flag_country = FLAG_COUNTRY_HINTS.get(vehicle_flag_code)

    return vehicle_type, vehicle_name, vehicle_flag_code, vehicle_flag_country, vehicle_call_sign


def parse_assistance(raw):
    s = collapsed_record(raw)
    section = section_between_any(
        s,
        ["Type of assistance provided by Cospas-Sarsat:", "Type of assistance provided:",
         "Assistance provided by Cospas-Sarsat:", "Number Persons Involved:", "Number of Persons Involved:", "Persons Involved:"],
        ["Details of Incident:", "Incident Details:", "Details:"]
    ) or ""

    m = re.search(r"(?:Number Persons Involved|Number of Persons Involved|Persons Involved):\s*([0-9]+(?:\.[0-9]+)?)", section, re.I)
    number_persons_involved = to_float(m.group(1)) if m else None

    m = re.search(r"(?:Cospas-Sarsat Assisted Saves|Cospas Sarsat Assisted Saves|Assisted Saves):\s*([0-9]+(?:\.[0-9]+)?)", section, re.I)
    cospas_sarsat_assisted_saves = to_float(m.group(1)) if m else None

    assistance_type = None
    section_l = section.lower()
    for value in ASSISTANCE_VALUES:
        if value.lower() in section_l:
            assistance_type = "supporting data" if value == "supported data" else value
            break

    if assistance_type is None:
        m = re.search(r"(?:Cospas-Sarsat Assisted Saves|Cospas Sarsat Assisted Saves|Assisted Saves):\s*[0-9]+(?:\.[0-9]+)?\s*([A-Za-z ]+)$", section, re.I)
        if m:
            candidate = clean_value_until_label(m.group(1))
            if candidate and candidate.lower() in [v.lower() for v in ASSISTANCE_VALUES]:
                assistance_type = "supporting data" if candidate.lower() == "supported data" else candidate.lower()

    return assistance_type, number_persons_involved, cospas_sarsat_assisted_saves


def parse_details(raw):
    s = collapsed_record(raw)
    m = re.search(r"(?:Details of Incident|Incident Details|Details):\s*(.*)", s, flags=re.S | re.I)
    return clean_value_until_label(m.group(1)) if m else None


In [ ]:
def parse_record(raw: str, source_pdf: str, source_pdf_page: int | None, report_year: int | None) -> dict | None:
    s = collapsed_record(raw)
    m = re.search(r"Reporting MCC:\s*(.*?)\s+Event ID:\s*([A-Z]{1,3}-\d{2,4})", s, flags=re.S | re.I)
    reporting_mcc = None
    if m:
        reporting_mcc = norm_ws(cleanup_headers(m.group(1)))
        if reporting_mcc:
            reporting_mcc = reporting_mcc.split()[-1]
        event_id = norm_ws(m.group(2))
    else:
        m = re.search(r"Event ID:\s*([A-Z]{1,3}-\d{2,4})", s, flags=re.I)
        if not m:
            return None
        event_id = norm_ws(m.group(1))
        mmcc = re.search(r"(?:Reporting MCC|MCC|Reporting RCC|RCC):\s*([A-Z0-9]+)", s, flags=re.I)
        reporting_mcc = mmcc.group(1).upper() if mmcc else None

    m = re.search(
        r"(?:Date of Incident(?: \(YYYY-MM-DD\))?|Date of incident|Incident Date):\s*([0-9]{4}-[0-9]{2}-[0-9]{2}|\d{1,2}-[A-Za-z]{3,9}-\d{2,4}|\d{1,2}\s+[A-Za-z]{3,9}\s+\d{2,4})",
        s,
        flags=re.I,
    )
    incident_date_raw = norm_ws(m.group(1)) if m else None
    incident_date = parse_date(incident_date_raw)

    m = re.search(r"Alert time in GMT(?: \(HH:MM\))?:\s*([0-9]{1,2}:[0-9]{2})", s, flags=re.I)
    alert_time_gmt = m.group(1) if m else None

    mlat = re.search(r"Latitude:\s*(-?\d+(?:\.\d+)?)\s+degrees\s+(-?\d+(?:\.\d+)?)\s+minutes\s+(North|South)", s, re.I)
    mlon = re.search(
        r"Longitude:\s*(-?\d+(?:\.\d+)?)\s+degrees\s+(-?\d+(?:\.\d+)?)\s+minutes\s+(East|West)\s*(Aviation|Maritime|Land)?",
        s,
        re.I,
    )

    latitude_degrees = latitude_minutes = latitude_hemisphere = None
    longitude_degrees = longitude_minutes = longitude_hemisphere = None
    incident_type = None
    if mlat:
        latitude_degrees = int(float(mlat.group(1)))
        latitude_minutes = float(mlat.group(2))
        latitude_hemisphere = mlat.group(3).title()
    if mlon:
        longitude_degrees = int(float(mlon.group(1)))
        longitude_minutes = float(mlon.group(2))
        longitude_hemisphere = mlon.group(3).title()
        if mlon.group(4):
            incident_type = mlon.group(4).title()

    # Decimal-coordinate fallback sometimes appears in older Annex-C-only PDFs.
    if latitude_degrees is None:
        m = re.search(r"Latitude:\s*(-?\d+(?:\.\d+)?)\s*(North|South)?", s, re.I)
        if m:
            val = float(m.group(1))
            hemi = m.group(2).title() if m.group(2) else ("South" if val < 0 else "North")
            latitude_degrees = int(abs(val))
            latitude_minutes = round((abs(val) - latitude_degrees) * 60, 6)
            latitude_hemisphere = hemi
    if longitude_degrees is None:
        m = re.search(r"Longitude:\s*(-?\d+(?:\.\d+)?)\s*(East|West)?", s, re.I)
        if m:
            val = float(m.group(1))
            hemi = m.group(2).title() if m.group(2) else ("West" if val < 0 else "East")
            longitude_degrees = int(abs(val))
            longitude_minutes = round((abs(val) - longitude_degrees) * 60, 6)
            longitude_hemisphere = hemi

    if not incident_type:
        m = re.search(r"Type of incident:\s*(Aviation|Maritime|Land)", s, re.I)
        if m:
            incident_type = m.group(1).title()

    if incident_type not in {"Aviation", "Maritime", "Land", None}:
        m = re.search(r"\b(Aviation|Maritime|Land)\b", s, re.I)
        incident_type = m.group(1).title() if m else None

    beacon_type, beacon_hex, beacon_frequency_mhz, beacon_country_code, beacon_country_name, valid_reg = extract_beacon(raw)
    vehicle_type, vehicle_name, vehicle_flag_code, vehicle_flag_country, vehicle_call_sign = parse_vehicle(raw)
    assistance_type, number_persons_involved, cospas_sarsat_assisted_saves = parse_assistance(raw)

    return {
        "event_id": event_id,
        "reporting_mcc": reporting_mcc,
        "incident_date": incident_date,
        "incident_date_raw": incident_date_raw,
        "alert_time_gmt": alert_time_gmt,
        "incident_type": incident_type,
        "location_description": extract_location(raw),
        "latitude_degrees": latitude_degrees,
        "latitude_minutes": latitude_minutes,
        "latitude_hemisphere": latitude_hemisphere,
        "longitude_degrees": longitude_degrees,
        "longitude_minutes": longitude_minutes,
        "longitude_hemisphere": longitude_hemisphere,
        "latitude_decimal": decimal_coord(latitude_degrees, latitude_minutes, latitude_hemisphere),
        "longitude_decimal": decimal_coord(longitude_degrees, longitude_minutes, longitude_hemisphere),
        "beacon_type": beacon_type,
        "beacon_hex": beacon_hex,
        "beacon_frequency_mhz": beacon_frequency_mhz,
        "beacon_country_code": beacon_country_code,
        "beacon_country_name": beacon_country_name,
        "valid_beacon_registration_available": valid_reg,
        "vehicle_type": vehicle_type,
        "vehicle_name": vehicle_name,
        "vehicle_flag_code": vehicle_flag_code,
        "vehicle_flag_country": vehicle_flag_country,
        "vehicle_call_sign": vehicle_call_sign,
        "assistance_type": assistance_type,
        "number_persons_involved": number_persons_involved,
        "cospas_sarsat_assisted_saves": cospas_sarsat_assisted_saves,
        "incident_details": parse_details(raw),
        "source_pdf": source_pdf,
        "source_pdf_page": source_pdf_page,
        "report_year": report_year,
        "raw_record_text": cleanup_record_text(raw),
    }


In [ ]:
def record_start_matches(annex: str):
    """Find likely record starts. Prefer Reporting MCC + Event ID; fall back to Event ID-only for older Annex-C PDFs."""
    pair_pat = re.compile(r"Reporting MCC:\s*.{0,250}?Event ID:\s*[A-Z]{1,3}-\d{2,4}", re.S | re.I)
    matches = list(pair_pat.finditer(annex))
    if matches:
        return [(m.start(), m.group(0)) for m in matches]

    # Older Annex-C-only files may not have Reporting MCC in the same way.
    event_pat = re.compile(r"Event ID:\s*[A-Z]{1,3}-\d{2,4}", re.I)
    return [(m.start(), m.group(0)) for m in event_pat.finditer(annex)]


def parse_pdf_records(pdf_path: Path) -> tuple[list[dict], list[dict]]:
    """Parse all Annex C records from a PDF. Returns records and page-end header diagnostic rows."""
    text_path = extract_text_with_pages(pdf_path)
    raw_text = text_path.read_text(encoding="utf-8", errors="replace")
    report_year = infer_report_year(raw_text, pdf_path)

    # Add explicit page markers. pdftotext separates pages using form feed characters.
    pages = raw_text.split("\f")
    marked = []
    for page_no, page_text in enumerate(pages, start=1):
        marked.append(f"\n[[PDF_PAGE:{page_no}]]\n{page_text}")
    full = "".join(marked)

    # Start at Annex C if the document contains a full report; otherwise use whole text.
    annex_positions = [m.start() for m in re.finditer(r"ANNEX\s+C|Annex\s+C", full)]
    first_record_candidates = [p for p in [full.find("Reporting MCC:"), full.find("Event ID:")] if p >= 0]
    if not first_record_candidates:
        raise RuntimeError(f"No `Reporting MCC:` or `Event ID:` labels found in {pdf_path.name}")
    first_record = min(first_record_candidates)
    annex_start = max([p for p in annex_positions if p <= first_record], default=first_record)
    annex = full[annex_start:]

    starts = record_start_matches(annex)
    if not starts:
        raise RuntimeError(f"No incident record starts found in {pdf_path.name}")

    records = []
    page_end_instances = []

    for idx, (start, start_text) in enumerate(starts):
        end = starts[idx + 1][0] if idx + 1 < len(starts) else len(annex)
        chunk = annex[start:end]

        abs_start = annex_start + start
        markers = re.findall(r"\[\[PDF_PAGE:(\d+)\]\]", full[:abs_start])
        source_pdf_page = int(markers[-1]) if markers else None

        rec = parse_record(chunk, source_pdf=pdf_path.name, source_pdf_page=source_pdf_page, report_year=report_year)
        if rec:
            records.append(rec)

            # Diagnostic: event header at bottom of a page, body likely on next page.
            head = chunk[:500]
            if "[[PDF_PAGE:" in chunk[:1200]:
                page_end_instances.append({
                    "source_pdf": pdf_path.name,
                    "report_year": report_year,
                    "event_id": rec["event_id"],
                    "reporting_mcc": rec["reporting_mcc"],
                    "source_pdf_page": source_pdf_page,
                    "next_page_marker_seen_in_record_start": True,
                    "record_start_preview": norm_ws(cleanup_headers(head)),
                })

    return records, page_end_instances


In [ ]:

# v4 compatibility overrides for older Annex-C-only PDFs (2007-2014 style)
OLD_DATE_LABEL_RE = r"(?:Date\s+of\s+Incident|Date\s+of\s+incident|Incident\s+Date|Date)"
OLD_DATE_VALUE_RE = r"(?:[A-Za-z]{3,9}[- ]\d{1,2}[- ]\d{2,4}|\d{1,2}[- ][A-Za-z]{3,9}[- ]\d{2,4}|\d{4}-\d{2}-\d{2})"
# These older files often do not have Event ID labels. Their records start at `Date of Incident:`
# and use a different form layout: `Persons Involved`, `Persons Rescued`, `Beacon Hex ID`,
# `Cospas-Sarsat provided`, etc. The functions below override the earlier parser where needed.

def parse_date(raw):
    raw = norm_ws(raw)
    if not raw:
        return None

    # Modern and older formats seen in C/S R.007 Annex C.
    for fmt in [
        "%Y-%m-%d", "%d-%b-%Y", "%d-%b-%y", "%d %b %Y", "%d %b %y",
        "%d-%B-%Y", "%d-%B-%y", "%B-%d-%Y", "%B-%d-%y", "%B %d %Y", "%B %d %y",
        "%b-%d-%Y", "%b-%d-%y", "%b %d %Y", "%b %d %y",
    ]:
        try:
            return datetime.strptime(raw, fmt).date().isoformat()
        except Exception:
            pass

    month_map = {
        "jan": 1, "january": 1, "feb": 2, "february": 2, "mar": 3, "march": 3,
        "apr": 4, "april": 4, "may": 5, "jun": 6, "june": 6, "jul": 7, "july": 7,
        "aug": 8, "august": 8, "sep": 9, "sept": 9, "september": 9,
        "oct": 10, "october": 10, "nov": 11, "november": 11, "dec": 12, "december": 12,
    }
    # dd-Mon-yy
    m = re.match(r"(\d{1,2})[- ]([A-Za-z]{3,9})[- ](\d{2,4})", raw)
    if m:
        d, mo, y = m.groups()
    else:
        # Month-dd-yy
        m = re.match(r"([A-Za-z]{3,9})[- ](\d{1,2})[- ](\d{2,4})", raw)
        if not m:
            return None
        mo, d, y = m.groups()
    mo_key = mo.lower()
    if mo_key not in month_map:
        mo_key = mo_key[:3]
    if mo_key not in month_map:
        return None
    yy = int(y)
    if yy < 100:
        yy += 2000
    return f"{yy:04d}-{month_map[mo_key]:02d}-{int(d):02d}"


def first_nonempty_line(text):
    for line in (text or "").splitlines():
        if line.strip():
            return line.strip()
    return None


def old_column_value(header_line: str, value_line: str, label: str, next_labels: list[str]):
    """Extract fixed-width value under a label from older Annex C forms."""
    if not header_line or not value_line or label not in header_line:
        return None
    start = header_line.find(label)
    stops = [header_line.find(nl) for nl in next_labels if header_line.find(nl) > start]
    end = min(stops) if stops else len(value_line)
    return clean_value_until_label(value_line[start:end].strip())


def parse_old_coord(location_line: str):
    """Parse old coordinates like 45°08' N 068°38' W."""
    if not location_line:
        return None, None, None, None, None, None
    m = re.search(
        r"(\d{1,3})\s*[°º]\s*(\d{1,2})\s*'?\s*([NS])\s+(\d{1,3})\s*[°º]\s*(\d{1,2})\s*'?\s*([EW])",
        location_line,
        flags=re.I,
    )
    if not m:
        return None, None, None, None, None, None
    lat_d, lat_m, lat_h, lon_d, lon_m, lon_h = m.groups()
    return int(lat_d), float(lat_m), {"N": "North", "S": "South"}[lat_h.upper()], int(lon_d), float(lon_m), {"E": "East", "W": "West"}[lon_h.upper()]


def old_country_code_from_name(name):
    # Limited reverse map for compatibility. Older reports generally do not print a 3-digit beacon country code.
    if not name:
        return None
    n = norm_ws(name)
    reverse = {v: k for k, v in COUNTRY_CODE_MAP.items()}
    aliases = {
        "United States": "366", "USA": "366", "Canada": "316", "Australia": "503", "New Zealand": "512",
        "France": "227", "United Kingdom": "234", "Russian Federation": "273", "Russia": "273",
    }
    return aliases.get(n) or reverse.get(n)


def extract_old_label_value(raw: str, label: str, stop_labels: list[str] | None = None):
    """Line-aware label extraction for older documents. Handles `Label: value` and `Label:\nvalue`."""
    stop_labels = stop_labels or FIELD_LABELS
    lines = raw.splitlines()
    for i, line in enumerate(lines):
        if label.lower() not in line.lower():
            continue
        # value on same line after label
        pos = line.lower().find(label.lower())
        tail = line[pos + len(label):].strip()
        if tail:
            # If another label appears on same line, keep only before it.
            cut = len(tail)
            for sl in stop_labels:
                p = tail.lower().find(sl.lower())
                if p >= 0:
                    cut = min(cut, p)
            val = tail[:cut].strip()
            if val:
                return clean_value_until_label(val)
        # value on following non-empty line, unless following line is another label row
        for j in range(i + 1, min(i + 4, len(lines))):
            candidate = lines[j].strip()
            if not candidate:
                continue
            if any(candidate.lower().startswith(sl.lower()) for sl in stop_labels):
                return None
            return clean_value_until_label(candidate)
    return None


def parse_old_annex_record(raw: str, source_pdf: str, source_pdf_page: int | None, report_year: int | None, seq: int) -> dict | None:
    """Parse older Annex C records that have Date of Incident but no Event ID."""
    raw_clean = cleanup_headers(strip_page_markers(raw or ""))
    lines = [ln.rstrip() for ln in raw_clean.splitlines()]
    compact = collapsed_record(raw_clean)

    mdate = re.search(OLD_DATE_LABEL_RE + r"\s*:\s*(" + OLD_DATE_VALUE_RE + r")", compact, flags=re.I)
    incident_date_raw = norm_ws(mdate.group(1)) if mdate else None
    incident_date = parse_date(incident_date_raw)
    if not incident_date and not incident_date_raw:
        return None

    mmcc = re.search(r"Reporting MCC:\s*([A-Z0-9]+)", compact, flags=re.I)
    reporting_mcc = mmcc.group(1).upper() if mmcc else None

    mp = re.search(r"Persons Involved:\s*(\d+(?:\.\d+)?)", compact, flags=re.I)
    number_persons_involved = to_float(mp.group(1)) if mp else None
    mr = re.search(r"Persons Rescued:\s*(\d+(?:\.\d+)?)", compact, flags=re.I)
    cospas_sarsat_assisted_saves = to_float(mr.group(1)) if mr else None

    # Locate key header/value row pairs.
    beacon_hex = beacon_type = beacon_frequency_mhz = beacon_country_name = None
    vehicle_type = vehicle_name = vehicle_call_sign = vehicle_flag_country = incident_type = None
    coord_line = None
    seen_beacon_column = False
    seen_vehicle_column = False

    for idx, line in enumerate(lines):
        if "Location:" in line and coord_line is None:
            # Coordinates can be on same line after Location:
            tail = line.split("Location:", 1)[1].strip() if "Location:" in line else ""
            if any(h in tail for h in [" N", " S", " E", " W", "°"]):
                coord_line = tail
            elif idx + 1 < len(lines):
                coord_line = lines[idx + 1].strip()

        if "Beacon Hex ID:" in line and "Type of beacon:" in line:
            seen_beacon_column = True
            value_line = lines[idx + 1] if idx + 1 < len(lines) else ""
            beacon_hex = old_column_value(line, value_line, "Beacon Hex ID:", ["Type of beacon:", "Beacon Frequency:", "Beacon Country:"])
            beacon_type = old_column_value(line, value_line, "Type of beacon:", ["Beacon Frequency:", "Beacon Country:"])
            freq_raw = old_column_value(line, value_line, "Beacon Frequency:", ["Beacon Country:"])
            m = re.search(r"(\d+(?:\.\d+)?)", freq_raw or "")
            beacon_frequency_mhz = float(m.group(1)) if m else None
            beacon_country_name = old_column_value(line, value_line, "Beacon Country:", [])

        if "Type of incident:" in line and "Vehicle:" in line:
            seen_vehicle_column = True
            value_line = lines[idx + 1] if idx + 1 < len(lines) else ""
            incident_type = old_column_value(line, value_line, "Type of incident:", ["Vehicle:", "Vehicle Name:", "Call Sign:", "Country Flag:"])
            vehicle_type = old_column_value(line, value_line, "Vehicle:", ["Vehicle Name:", "Call Sign:", "Country Flag:"])
            vehicle_name = old_column_value(line, value_line, "Vehicle Name:", ["Call Sign:", "Country Flag:"])
            vehicle_call_sign = old_column_value(line, value_line, "Call Sign:", ["Country Flag:"])
            vehicle_flag_country = old_column_value(line, value_line, "Country Flag:", [])

    # Fallbacks from line labels if column extraction did not catch something.
    # Important: in older PDFs the headers are multi-column rows, so a naive `Vehicle Name:` fallback
    # can accidentally return the whole value row (`Aviation C-172 Canada`). Only use label fallback for
    # vehicle fields when no fixed-width vehicle row was detected.
    beacon_hex = beacon_hex or extract_old_label_value(raw_clean, "Beacon Hex ID:")
    beacon_type = beacon_type or extract_old_label_value(raw_clean, "Type of beacon:")
    if beacon_frequency_mhz is None:
        freq_raw = extract_old_label_value(raw_clean, "Beacon Frequency:")
        mf = re.search(r"(\d+(?:\.\d+)?)", freq_raw or "")
        beacon_frequency_mhz = float(mf.group(1)) if mf else None
    beacon_country_name = beacon_country_name or extract_old_label_value(raw_clean, "Beacon Country:")
    if not seen_vehicle_column:
        vehicle_flag_country = vehicle_flag_country or extract_old_label_value(raw_clean, "Country Flag:")
        incident_type = incident_type or extract_old_label_value(raw_clean, "Type of incident:")
        vehicle_type = vehicle_type or extract_old_label_value(raw_clean, "Vehicle:")
        vehicle_name = vehicle_name or extract_old_label_value(raw_clean, "Vehicle Name:")
        vehicle_call_sign = vehicle_call_sign or extract_old_label_value(raw_clean, "Call Sign:")

    # Assistance and registration.
    ma = re.search(r"Cospas-Sarsat provided:\s*(.*?)(?:\s+Is Beacon Registered\?|\n|$)", raw_clean, flags=re.I)
    assistance_type = clean_value_until_label(ma.group(1)) if ma else None
    if assistance_type:
        assistance_type = assistance_type.lower()
        if assistance_type == "supported data":
            assistance_type = "supporting data"

    mreg = re.search(r"Is Beacon Registered\?\s*(Yes|No)", raw_clean, flags=re.I)
    valid_reg = None if not mreg else (mreg.group(1).lower() == "yes")

    lat_d, lat_m, lat_h, lon_d, lon_m, lon_h = parse_old_coord(coord_line or compact)

    # Details: first line after Details of Incident is the location description; the rest is details.
    details = None
    location_description = None
    mdet = re.search(r"Details of Incident:\s*(.*)", raw_clean, flags=re.S | re.I)
    if mdet:
        detail_lines = [ln.strip() for ln in mdet.group(1).splitlines() if ln.strip()]
        if detail_lines:
            location_description = clean_value_until_label(detail_lines[0])
            details = norm_ws(" ".join(detail_lines[1:])) if len(detail_lines) > 1 else None

    # If details location failed, use non-coordinate Location text if present.
    if not location_description:
        loc_raw = extract_location(raw_clean)
        if loc_raw and not re.search(r"\d{1,3}\s*[°º]", loc_raw):
            location_description = loc_raw

    vehicle_flag_code = None
    if vehicle_flag_country:
        # Reverse from known hints where possible, otherwise keep code null for old docs.
        for code, name in FLAG_COUNTRY_HINTS.items():
            if name and vehicle_flag_country.lower() == name.lower():
                vehicle_flag_code = code
                break
        if vehicle_flag_country.upper() == "USA":
            vehicle_flag_code = "USA"
            vehicle_flag_country = "United States"

    beacon_country_code = old_country_code_from_name(beacon_country_name)

    return {
        "event_id": f"{report_year or 'unknown'}-{seq:04d}",
        "reporting_mcc": reporting_mcc,
        "incident_date": incident_date,
        "incident_date_raw": incident_date_raw,
        "alert_time_gmt": None,
        "incident_type": incident_type.title() if incident_type and incident_type.lower() in {"aviation", "maritime", "land"} else incident_type,
        "location_description": location_description,
        "latitude_degrees": lat_d,
        "latitude_minutes": lat_m,
        "latitude_hemisphere": lat_h,
        "longitude_degrees": lon_d,
        "longitude_minutes": lon_m,
        "longitude_hemisphere": lon_h,
        "latitude_decimal": decimal_coord(lat_d, lat_m, lat_h),
        "longitude_decimal": decimal_coord(lon_d, lon_m, lon_h),
        "beacon_type": beacon_type.upper() if beacon_type else None,
        "beacon_hex": beacon_hex.upper() if beacon_hex and re.fullmatch(r"[A-Fa-f0-9]{5,}", beacon_hex) else None,
        "beacon_frequency_mhz": beacon_frequency_mhz,
        "beacon_country_code": beacon_country_code,
        "beacon_country_name": beacon_country_name,
        "valid_beacon_registration_available": valid_reg,
        "vehicle_type": vehicle_type,
        "vehicle_name": vehicle_name,
        "vehicle_flag_code": vehicle_flag_code,
        "vehicle_flag_country": vehicle_flag_country,
        "vehicle_call_sign": vehicle_call_sign,
        "assistance_type": assistance_type,
        "number_persons_involved": number_persons_involved,
        "cospas_sarsat_assisted_saves": cospas_sarsat_assisted_saves,
        "incident_details": details,
        "source_pdf": source_pdf,
        "source_pdf_page": source_pdf_page,
        "report_year": report_year,
        "raw_record_text": cleanup_record_text(raw_clean),
    }


def record_start_matches(annex: str):
    """Find likely modern record starts. Old Date-of-Incident-only records are handled separately."""
    pair_pat = re.compile(r"Reporting MCC:\s*.{0,250}?Event ID:\s*[A-Z]{1,3}-\d{2,4}", re.S | re.I)
    matches = list(pair_pat.finditer(annex))
    if matches:
        return [(m.start(), m.group(0), "modern_pair") for m in matches]

    event_pat = re.compile(r"Event ID:\s*[A-Z]{1,3}-\d{2,4}", re.I)
    matches = list(event_pat.finditer(annex))
    if matches:
        return [(m.start(), m.group(0), "modern_event") for m in matches]

    date_pat = re.compile(OLD_DATE_LABEL_RE + r"\s*:\s*" + OLD_DATE_VALUE_RE, re.I)
    matches = list(date_pat.finditer(annex))
    return [(m.start(), m.group(0), "old_date") for m in matches]


def parse_pdf_records(pdf_path: Path) -> tuple[list[dict], list[dict]]:
    """Parse all Annex C records from a PDF. Supports modern Event-ID forms and older Date-of-Incident-only forms."""
    text_path = extract_text_with_pages(pdf_path)
    raw_text = text_path.read_text(encoding="utf-8", errors="replace")
    report_year = infer_report_year(raw_text, pdf_path)

    pages = raw_text.split("\f")
    marked = []
    for page_no, page_text in enumerate(pages, start=1):
        marked.append(f"\n[[PDF_PAGE:{page_no}]]\n{page_text}")
    full = "".join(marked)

    # Prefer the first real incident list record, not the summary/table section.
    first_modern_candidates = [p for p in [full.find("Reporting MCC:"), full.find("Event ID:")] if p >= 0]
    first_old_candidates = [m.start() for m in re.finditer(OLD_DATE_LABEL_RE + r"\s*:\s*" + OLD_DATE_VALUE_RE, full, flags=re.I)]
    candidates = first_modern_candidates + first_old_candidates
    if not candidates:
        raise RuntimeError(f"No incident record starts found in {pdf_path.name}; looked for Reporting MCC, Event ID, and Date of Incident")
    first_record = min(candidates)

    annex_positions = [m.start() for m in re.finditer(r"ANNEX\s+C|Annex\s+C|LIST OF SEARCH AND RESCUE EVENTS", full, flags=re.I)]
    annex_start = max([p for p in annex_positions if p <= first_record], default=first_record)
    annex = full[annex_start:]

    starts = record_start_matches(annex)
    if not starts:
        raise RuntimeError(f"No incident record starts found in {pdf_path.name}")

    records = []
    page_end_instances = []

    for idx, (start, start_text, start_kind) in enumerate(starts):
        end = starts[idx + 1][0] if idx + 1 < len(starts) else len(annex)
        chunk = annex[start:end]
        abs_start = annex_start + start
        markers = re.findall(r"\[\[PDF_PAGE:(\d+)\]\]", full[:abs_start])
        source_pdf_page = int(markers[-1]) if markers else None

        if start_kind == "old_date":
            rec = parse_old_annex_record(chunk, source_pdf=pdf_path.name, source_pdf_page=source_pdf_page, report_year=report_year, seq=idx + 1)
        else:
            rec = parse_record(chunk, source_pdf=pdf_path.name, source_pdf_page=source_pdf_page, report_year=report_year)

        if rec:
            records.append(rec)
            if start_kind != "old_date" and "[[PDF_PAGE:" in chunk[:1200]:
                page_end_instances.append({
                    "source_pdf": pdf_path.name,
                    "report_year": report_year,
                    "event_id": rec["event_id"],
                    "reporting_mcc": rec["reporting_mcc"],
                    "source_pdf_page": source_pdf_page,
                    "raw_record_head": cleanup_record_text(chunk[:800]),
                })

    return records, page_end_instances


def is_canada_record(record: dict) -> bool:
    """Canada filter. Modern records use CA-* Event IDs; older records have no Event ID, so use Canadian MCC/location/registration signals."""
    eid = str(record.get("event_id") or "")
    if eid.startswith("CA-"):
        return True
    text_fields = [
        record.get("reporting_mcc"),
        record.get("location_description"),
        record.get("beacon_country_name"),
        record.get("vehicle_flag_country"),
        record.get("incident_details"),
    ]
    text = " | ".join(str(x) for x in text_fields if x)
    if record.get("reporting_mcc") == "CMCC":
        return True
    if re.search(r"\bCanada\b|\bCanadian\b", text, flags=re.I):
        return True
    if record.get("beacon_country_code") == "316":
        return True
    if record.get("vehicle_flag_code") == "CAN":
        return True
    return False


# v5 overrides: better support for 2012/older Annex-C extracted text where labels lack colons
# and pdftotext emits form rows as line-label/value sequences rather than true fixed columns.
COUNTRY_NAME_TO_CODE_EXTRA = {
    "Canada": "316", "United States": "366", "USA": "366", "Australia": "503", "New Zealand": "512",
    "France": "227", "United Kingdom": "234", "Russian Federation": "273", "Russia": "273",
    "Germany": "262", "Italy": "247", "Netherlands": "244", "Norway": "257", "Spain": "224",
    "China": "412", "China (P.R. of)": "412", "Japan": "440", "Chile": "725", "South Africa": "710",
}
COUNTRY_NAMES_LONG_FIRST = sorted(
    set(COUNTRY_CODE_MAP.values()) | set(COUNTRY_NAME_TO_CODE_EXTRA.keys()) |
    {"United States", "Russian Federation", "New Zealand", "Czech Republic", "United Kingdom"},
    key=len,
    reverse=True,
)


def split_country_code_name(value):
    """Split older Beacon Country values like `316 Canada` or `503 Australia`."""
    value = norm_ws(value)
    if not value:
        return None, None
    m = re.match(r"^(\d{3})\s+(.+)$", value)
    if m:
        code, name = m.groups()
        return code, clean_value_until_label(name)
    if re.fullmatch(r"\d{3}", value):
        return value, COUNTRY_CODE_MAP.get(value)
    code = old_country_code_from_name(value) or COUNTRY_NAME_TO_CODE_EXTRA.get(value)
    return code, value


def split_trailing_country(value):
    """Split values like `C-FGBX Canada`, `N/A United States`, or `FXYH4 Australia`."""
    value = norm_ws(value)
    if not value:
        return None, None
    for country in COUNTRY_NAMES_LONG_FIRST:
        if value == country:
            return None, country
        if value.lower().endswith(" " + country.lower()):
            left = value[: -len(country)].strip()
            return (left or None), country
    return value, None


def parse_old_coord(location_line: str):
    """Parse old coordinates like 45°08' N 068°38' W or 20°27' South 150°35' East."""
    if not location_line:
        return None, None, None, None, None, None
    text = norm_ws(location_line)
    # Normalize full hemisphere words to letters for simpler parsing.
    text = re.sub(r"\bNorth\b", "N", text, flags=re.I)
    text = re.sub(r"\bSouth\b", "S", text, flags=re.I)
    text = re.sub(r"\bEast\b", "E", text, flags=re.I)
    text = re.sub(r"\bWest\b", "W", text, flags=re.I)
    # Some older text has a leading minus and a hemisphere word. Hemisphere wins.
    m = re.search(
        r"-?(\d{1,3})\s*[°º]\s*(\d{1,2})\s*' ?\s*([NS])\s+-?(\d{1,3})\s*[°º]\s*(\d{1,2})\s*' ?\s*([EW])",
        text,
        flags=re.I,
    )
    if not m:
        return None, None, None, None, None, None
    lat_d, lat_m, lat_h, lon_d, lon_m, lon_h = m.groups()
    return (
        int(lat_d), float(lat_m), {"N": "North", "S": "South"}[lat_h.upper()],
        int(lon_d), float(lon_m), {"E": "East", "W": "West"}[lon_h.upper()],
    )


def extract_old_label_value(raw: str, label: str, stop_labels: list[str] | None = None):
    """Line-aware label extraction for old files where labels may be `Label value` or `Label: value`."""
    stop_labels = stop_labels or FIELD_LABELS
    lines = raw.splitlines()
    label_l = label.lower().rstrip(":")
    for i, line in enumerate(lines):
        line_s = line.strip()
        if not line_s:
            continue
        lower = line_s.lower()
        # Accept `Reporting MCC AUMCC` as well as `Reporting MCC: AUMCC`.
        if not (lower.startswith(label_l + ":") or lower.startswith(label_l + " ") or lower == label_l):
            continue
        tail = re.sub(r"^" + re.escape(label_l) + r"\s*: ?", "", line_s, flags=re.I).strip()
        if tail and tail.lower() != label_l:
            cut = len(tail)
            for sl in stop_labels:
                sl_l = sl.lower().rstrip(":")
                p = tail.lower().find(sl_l)
                if p > 0:
                    cut = min(cut, p)
            val = tail[:cut].strip()
            if val:
                return clean_value_until_label(val)
        for j in range(i + 1, min(i + 5, len(lines))):
            candidate = lines[j].strip()
            if not candidate:
                continue
            cand_lower = candidate.lower()
            if any(cand_lower.startswith(sl.lower().rstrip(":")) for sl in stop_labels):
                return None
            return clean_value_until_label(candidate)
    return None


def parse_old_annex_record(raw: str, source_pdf: str, source_pdf_page: int | None, report_year: int | None, seq: int) -> dict | None:
    """Parse older Annex C records that have Date of Incident but no Event ID."""
    raw_clean = cleanup_headers(strip_page_markers(raw or ""))
    lines = [ln.rstrip() for ln in raw_clean.splitlines()]
    compact = collapsed_record(raw_clean)

    mdate = re.search(OLD_DATE_LABEL_RE + r"\s*:\s*(" + OLD_DATE_VALUE_RE + r")", compact, flags=re.I)
    incident_date_raw = norm_ws(mdate.group(1)) if mdate else None
    incident_date = parse_date(incident_date_raw)
    if not incident_date and not incident_date_raw:
        return None

    mmcc = re.search(r"Reporting MCC\s*:?\s*([A-Z0-9]+)", compact, flags=re.I)
    reporting_mcc = mmcc.group(1).upper() if mmcc else None

    mp = re.search(r"Persons Involved\s*:?\s*(\d+(?:\.\d+)?)", compact, flags=re.I)
    number_persons_involved = to_float(mp.group(1)) if mp else None
    mr = re.search(r"Persons Rescued\s*:?\s*(\d+(?:\.\d+)?)", compact, flags=re.I)
    cospas_sarsat_assisted_saves = to_float(mr.group(1)) if mr else None

    beacon_hex = beacon_type = beacon_frequency_mhz = None
    beacon_country_code = beacon_country_name = None
    vehicle_type = vehicle_name = vehicle_call_sign = vehicle_flag_country = incident_type = None
    coord_line = None
    seen_beacon_column = False
    seen_vehicle_column = False

    for idx, line in enumerate(lines):
        if "Location:" in line and coord_line is None:
            tail = line.split("Location:", 1)[1].strip() if "Location:" in line else ""
            if any(h in tail for h in [" N", " S", " E", " W", "°", "North", "South", "East", "West"]):
                coord_line = tail
            elif idx + 1 < len(lines):
                coord_line = lines[idx + 1].strip()

        if "Beacon Hex ID:" in line and "Type of beacon:" in line:
            seen_beacon_column = True
            value_line = lines[idx + 1] if idx + 1 < len(lines) else ""
            beacon_hex = old_column_value(line, value_line, "Beacon Hex ID:", ["Type of beacon:", "Beacon Frequency:", "Beacon Country:"])
            beacon_type = old_column_value(line, value_line, "Type of beacon:", ["Beacon Frequency:", "Beacon Country:"])
            freq_raw = old_column_value(line, value_line, "Beacon Frequency:", ["Beacon Country:"])
            m = re.search(r"(\d+(?:\.\d+)?)", freq_raw or "")
            beacon_frequency_mhz = float(m.group(1)) if m else None
            country_raw = old_column_value(line, value_line, "Beacon Country:", [])
            beacon_country_code, beacon_country_name = split_country_code_name(country_raw)

        if "Type of incident:" in line and "Vehicle:" in line:
            seen_vehicle_column = True
            value_line = lines[idx + 1] if idx + 1 < len(lines) else ""
            incident_type = old_column_value(line, value_line, "Type of incident:", ["Vehicle:", "Vehicle Name:", "Call Sign:", "Country Flag:", "Vessel Country Flag"])
            vehicle_type = old_column_value(line, value_line, "Vehicle:", ["Vehicle Name:", "Call Sign:", "Country Flag:", "Vessel Country Flag"])
            vehicle_name = old_column_value(line, value_line, "Vehicle Name:", ["Call Sign:", "Country Flag:", "Vessel Country Flag"])
            vehicle_call_sign = old_column_value(line, value_line, "Call Sign:", ["Country Flag:", "Vessel Country Flag"])
            vehicle_flag_country = old_column_value(line, value_line, "Country Flag:", []) or old_column_value(line, value_line, "Vessel Country Flag", [])

    # Fallbacks from label/value sequences emitted by pdftotext.
    beacon_hex = beacon_hex or extract_old_label_value(raw_clean, "Beacon Hex ID")
    beacon_type = beacon_type or extract_old_label_value(raw_clean, "Type of beacon")
    if beacon_frequency_mhz is None:
        freq_raw = extract_old_label_value(raw_clean, "Beacon Frequency")
        mf = re.search(r"(\d+(?:\.\d+)?)", freq_raw or "")
        beacon_frequency_mhz = float(mf.group(1)) if mf else None
    if not beacon_country_name and not beacon_country_code:
        country_raw = extract_old_label_value(raw_clean, "Beacon Country")
        beacon_country_code, beacon_country_name = split_country_code_name(country_raw)

    if not seen_vehicle_column:
        incident_type = incident_type or extract_old_label_value(raw_clean, "Type of incident")
        vehicle_type = vehicle_type or extract_old_label_value(raw_clean, "Vehicle")
        vehicle_name = vehicle_name or extract_old_label_value(raw_clean, "Vehicle Name")
        vehicle_call_sign = vehicle_call_sign or extract_old_label_value(raw_clean, "Call Sign")
        vehicle_flag_country = vehicle_flag_country or extract_old_label_value(raw_clean, "Country Flag") or extract_old_label_value(raw_clean, "Vessel Country Flag")

    # If call sign contains the trailing flag country, split it out.
    if vehicle_call_sign and not vehicle_flag_country:
        vehicle_call_sign, vehicle_flag_country = split_trailing_country(vehicle_call_sign)
    elif vehicle_call_sign and vehicle_flag_country:
        maybe_call, maybe_country = split_trailing_country(vehicle_call_sign)
        if maybe_country and maybe_country.lower() == vehicle_flag_country.lower():
            vehicle_call_sign = maybe_call

    # Assistance and registration: older text often lacks a colon.
    ma = re.search(r"Cospas-Sarsat provided\s*:?\s*(.*?)(?:\s+Beacon Registration Data Available|\s+Is Beacon Registered\?|\n|$)", raw_clean, flags=re.I)
    assistance_type = clean_value_until_label(ma.group(1)) if ma else None
    if assistance_type:
        assistance_type = assistance_type.lower()
        if assistance_type in {"supported data", "support data"}:
            assistance_type = "supporting data"

    mreg = re.search(r"(?:Is Beacon Registered\?|Beacon Registration Data Available)\s*:?\s*(Yes|No)", raw_clean, flags=re.I)
    valid_reg = None if not mreg else (mreg.group(1).lower() == "yes")

    lat_d, lat_m, lat_h, lon_d, lon_m, lon_h = parse_old_coord(coord_line or compact)

    details = None
    location_description = None
    mdet = re.search(r"Details of Incident:\s*(.*)", raw_clean, flags=re.S | re.I)
    if mdet:
        detail_lines = [ln.strip() for ln in mdet.group(1).splitlines() if ln.strip()]
        if detail_lines:
            location_description = clean_value_until_label(detail_lines[0])
            details = norm_ws(" ".join(detail_lines[1:])) if len(detail_lines) > 1 else None

    if not location_description:
        loc_raw = extract_location(raw_clean)
        if loc_raw and not re.search(r"\d{1,3}\s*[°º]", loc_raw):
            location_description = loc_raw

    vehicle_flag_code = None
    if vehicle_flag_country:
        for code, name in FLAG_COUNTRY_HINTS.items():
            if name and vehicle_flag_country.lower() == name.lower():
                vehicle_flag_code = code
                break
        if vehicle_flag_country.upper() == "USA":
            vehicle_flag_code = "USA"
            vehicle_flag_country = "United States"
        elif vehicle_flag_country == "Canada":
            vehicle_flag_code = "CAN"
        elif vehicle_flag_country == "Australia":
            vehicle_flag_code = "AUS"
        elif vehicle_flag_country == "New Zealand":
            vehicle_flag_code = "NZL"

    if beacon_country_code and not beacon_country_name:
        beacon_country_name = COUNTRY_CODE_MAP.get(beacon_country_code)
    if beacon_country_name and not beacon_country_code:
        beacon_country_code = old_country_code_from_name(beacon_country_name) or COUNTRY_NAME_TO_CODE_EXTRA.get(beacon_country_name)

    return {
        "event_id": f"{report_year or 'unknown'}-{seq:04d}",
        "reporting_mcc": reporting_mcc,
        "incident_date": incident_date,
        "incident_date_raw": incident_date_raw,
        "alert_time_gmt": None,
        "incident_type": incident_type.title() if incident_type and incident_type.lower() in {"aviation", "maritime", "land"} else incident_type,
        "location_description": location_description,
        "latitude_degrees": lat_d,
        "latitude_minutes": lat_m,
        "latitude_hemisphere": lat_h,
        "longitude_degrees": lon_d,
        "longitude_minutes": lon_m,
        "longitude_hemisphere": lon_h,
        "latitude_decimal": decimal_coord(lat_d, lat_m, lat_h),
        "longitude_decimal": decimal_coord(lon_d, lon_m, lon_h),
        "beacon_type": beacon_type.upper() if beacon_type else None,
        "beacon_hex": beacon_hex.upper() if beacon_hex and re.fullmatch(r"[A-Fa-f0-9]{5,}|N/A", beacon_hex) else None,
        "beacon_frequency_mhz": beacon_frequency_mhz,
        "beacon_country_code": beacon_country_code,
        "beacon_country_name": beacon_country_name,
        "valid_beacon_registration_available": valid_reg,
        "vehicle_type": vehicle_type,
        "vehicle_name": vehicle_name,
        "vehicle_flag_code": vehicle_flag_code,
        "vehicle_flag_country": vehicle_flag_country,
        "vehicle_call_sign": vehicle_call_sign,
        "assistance_type": assistance_type,
        "number_persons_involved": number_persons_involved,
        "cospas_sarsat_assisted_saves": cospas_sarsat_assisted_saves,
        "incident_details": details,
        "source_pdf": source_pdf,
        "source_pdf_page": source_pdf_page,
        "report_year": report_year,
        "raw_record_text": cleanup_record_text(raw_clean),
    }


In [ ]:

# v7 final overrides for remaining older layouts: 2014 EVENT forms and 2007-2009 table layout.
# These definitions intentionally override parse_pdf_records from earlier cells.

MONTH_NUM = {
    "january": 1, "february": 2, "march": 3, "april": 4, "may": 5, "june": 6,
    "july": 7, "august": 8, "september": 9, "october": 10, "november": 11, "december": 12,
    "jan": 1, "feb": 2, "mar": 3, "apr": 4, "jun": 6, "jul": 7, "aug": 8, "sep": 9, "sept": 9,
    "oct": 10, "nov": 11, "dec": 12,
}

COUNTRY_CODE_BY_NAME = {
    "Canada": "316", "United States": "366", "USA": "366", "Australia": "503", "New Zealand": "512",
    "France": "227", "United Kingdom": "234", "UK": "234", "Russia": "273", "Russian Federation": "273",
    "Germany": "211", "Norway": "257", "Netherlands": "244", "The Netherlands": "244", "Italy": "247",
    "Spain": "224", "Argentina": "701", "Brazil": "710", "Chile": "725", "Japan": "431", "China": "412",
}

FLAG_CODE_BY_COUNTRY = {
    "Canada": "CAN", "United States": "USA", "USA": "USA", "Australia": "AUS", "New Zealand": "NZL",
    "France": "FRA", "United Kingdom": "GBR", "UK": "GBR", "Russia": "RUS", "Russian Federation": "RUS",
    "Germany": "DEU", "Norway": "NOR", "Netherlands": "NLD", "The Netherlands": "NLD", "Italy": "ITA",
    "Spain": "ESP", "Argentina": "ARG", "Brazil": "BRA", "Chile": "CHL", "Japan": "JPN", "China": "CHN",
}

KNOWN_COUNTRIES_FOR_OLD = sorted((set(COUNTRY_CODE_BY_NAME) | set(FLAG_CODE_BY_COUNTRY) | {
    "Tonga", "Bulgaria", "Morocco", "French Polynesia", "Denmark", "Sweden", "Germany", "Czech Republic",
    "Mongolia", "Austria", "Cambodia", "Namibia", "Panama", "Bahamas", "Mexico", "Greece", "Portugal",
    "South Africa", "Ireland", "Iceland", "Greenland", "Honduras", "Belgium", "Turkey", "Uruguay",
}), key=len, reverse=True)


def _first_match(pattern, text, flags=0, group=1):
    m = re.search(pattern, text or "", flags)
    return norm_ws(m.group(group)) if m else None


def _nonempty_lines(text):
    return [ln.rstrip() for ln in (text or "").splitlines() if ln.strip()]


def _strip_form_headers(text):
    text = re.sub(r"\[\[PDF_PAGE:\d+\]\]", "\n", text or "")
    text = re.sub(r"C/S Report on System Status.*?Annex C\s+Page\s+\d+", " ", text, flags=re.I)
    return text


def _line_after_label(lines, label_regex, max_lookahead=6):
    label_re = re.compile(label_regex, re.I)
    for i, ln in enumerate(lines):
        if label_re.search(ln):
            tail = label_re.sub("", ln).strip(" :\t")
            if tail:
                return clean_value_until_label(tail)
            for j in range(i + 1, min(i + max_lookahead + 1, len(lines))):
                cand = lines[j].strip()
                if not cand:
                    continue
                if re.search(r"(?:Date of Incident|Alert time|Location of Incident|Type of Incident|Beacon|Vehicle|Cospas|Persons|Details|EVENT|Reporting MCC)", cand, re.I):
                    continue
                return clean_value_until_label(cand)
    return None


def _value_on_same_or_next(raw, label_regex, max_lookahead=8):
    lines = _nonempty_lines(raw)
    return _line_after_label(lines, label_regex, max_lookahead=max_lookahead)


def _parse_event_form_coord(raw):
    # Form layout as emitted by pdftotext: degree line has hemisphere + degree values, minutes line has both minute values.
    m = re.search(
        r"Date of Incident.*?\n\s*(\d{4}-\d{2}-\d{2}).*?Latitude \(0-90\).*?Longitude\(0-180\).*?\n\s*(North|South)\s+(\d{1,3})\s+(East|West)\s+(\d{1,3}).*?Latitude minutes \(0-60\):\s*(\d{1,2}(?:\.\d+)?)\s+Longitude Minutes \(0-60\):\s*(\d{1,2}(?:\.\d+)?)",
        raw,
        flags=re.I | re.S,
    )
    if m:
        date, lat_h, lat_d, lon_h, lon_d, lat_m, lon_m = m.groups()
        return date, int(lat_d), float(lat_m), lat_h.title(), int(lon_d), float(lon_m), lon_h.title()
    return None, None, None, None, None, None, None


def _parse_location_incident_type_form(raw):
    lines = _nonempty_lines(raw)
    for i, ln in enumerate(lines):
        if re.search(r"Location of Incident\s*:", ln, re.I):
            # Prefer the value line immediately after the label row.
            for j in range(i + 1, min(i + 5, len(lines))):
                cand = lines[j].strip()
                if not cand or re.search(r"^(Type of beacon|Beacon|Vehicle|Cospas|Details|Alert time|Date of Incident)", cand, re.I):
                    continue
                parts = [p.strip() for p in re.split(r"\s{2,}", cand) if p.strip()]
                if parts:
                    loc = parts[0]
                    typ = None
                    for p in parts[1:] + [cand]:
                        mt = re.search(r"\b(Aviation|Maritime|Land)\b", p, re.I)
                        if mt:
                            typ = mt.group(1).title(); break
                    return clean_value_until_label(loc), typ
    loc = extract_location(raw)
    typ = extract_incident_type(raw)
    return loc, typ


def _parse_flag_and_call_form(raw):
    lines = _nonempty_lines(raw)
    flag_code = flag_country = call_sign = None
    for i, ln in enumerate(lines):
        if re.search(r"Vessel/Aircraft Flag", ln, re.I):
            window = "\n".join(lines[i:i+8])
            # Call sign is often on a line after `Call Sign:` or far right in the value row.
            mcall = re.search(r"Call Sign:\s*\n?\s*([A-Z0-9][A-Z0-9\-]{2,})", window, re.I)
            if mcall and not re.match(r"^(CAN|AUS|USA|NZL|FRA|CMB|0)$", mcall.group(1), re.I):
                call_sign = mcall.group(1).strip()
            for code, country in FLAG_COUNTRY_HINTS.items():
                if re.search(rf"\b{re.escape(code)}\b", window):
                    flag_code = code
                    flag_country = country
                    break
            if not flag_country:
                for country in KNOWN_COUNTRIES_FOR_OLD:
                    if re.search(rf"\b{re.escape(country)}\b", window, re.I):
                        flag_country = country
                        flag_code = FLAG_CODE_BY_COUNTRY.get(country)
                        break
            break
    return flag_code, flag_country, call_sign


def _parse_vehicle_form(raw):
    lines = _nonempty_lines(raw)
    vehicle_type = vehicle_name = None
    for i, ln in enumerate(lines):
        if re.search(r"Vehicle type", ln, re.I):
            # First non-label row after the header usually contains type and name columns.
            for j in range(i + 1, min(i + 6, len(lines))):
                cand = lines[j].strip()
                if not cand or re.search(r"^(Cospas|Persons|Details|Beacon|Vessel/Aircraft|Call Sign|EVENT|Date)", cand, re.I):
                    continue
                # Remove standalone country/flag rows; they belong to flag, not vehicle.
                if cand in KNOWN_COUNTRIES_FOR_OLD or re.fullmatch(r"[A-Z]{3}|0", cand):
                    continue
                parts = [p.strip() for p in re.split(r"\s{2,}", cand) if p.strip()]
                if len(parts) >= 2:
                    vehicle_type, vehicle_name = parts[0], parts[1]
                else:
                    vehicle_type = cand
                break
            break
    return clean_value_until_label(vehicle_type), clean_value_until_label(vehicle_name)


def _parse_event_form_record(raw, source_pdf, source_pdf_page, report_year, seq):
    raw_no_header = _strip_form_headers(raw)
    compact = collapsed_record(raw_no_header)
    evnum = _first_match(r"EVENT\s*:\s*(\d+)", compact, re.I)
    reporting_mcc = _first_match(r"Reporting MCC\s*:\s*([A-Z0-9]+)", compact, re.I)
    if not evnum and not reporting_mcc:
        return None

    incident_date_raw, lat_d, lat_m, lat_h, lon_d, lon_m, lon_h = _parse_event_form_coord(raw_no_header)
    incident_date = parse_date(incident_date_raw)
    alert_time = _first_match(r"Alert time in GMT:\s*\(HH:MM\)\s*([0-9]{1,2}:[0-9]{2})", compact, re.I)
    if not alert_time:
        alert_time = _first_match(r"\b([0-9]{1,2}:[0-9]{2})\b", compact, re.I)

    location_description, incident_type = _parse_location_incident_type_form(raw_no_header)
    beacon_type = _first_match(r"Type of beacon\s+([A-Z]{2,6})\b", compact, re.I)
    beacon_hex = _first_match(r"Beacon Hex ID:\s*([A-F0-9]{5,}|N/A)", compact, re.I)
    freq = _first_match(r"Beacon Frequency:\s*([0-9]+(?:\.[0-9]+)?)", compact, re.I)
    beacon_frequency_mhz = float(freq) if freq else None
    # Country code/name: code follows Beacon Country and name usually follows Yes/No on next line.
    bc_code = _first_match(r"Beacon Country:\s*(\d{3})", compact, re.I)
    bc_name = None
    if bc_code:
        bc_name = COUNTRY_CODE_MAP.get(bc_code)
    # fallback: look near Beacon Country for a country name
    if not bc_name:
        m = re.search(r"Beacon Country:\s*(?:\d{3})?\s*([A-Z][A-Za-z .()'-]+?)(?:\s+Vessel/Aircraft|\s+Call Sign|\s+Vehicle|$)", compact, re.I)
        if m:
            _, bc_name = split_country_code_name(m.group(1))
    if not bc_name:
        for country in KNOWN_COUNTRIES_FOR_OLD:
            if re.search(rf"Beacon Country:.*?\b{re.escape(country)}\b", compact, re.I):
                bc_name = country; break
    if bc_name and not bc_code:
        bc_code = COUNTRY_CODE_MAP.get(bc_name) or COUNTRY_CODE_BY_NAME.get(bc_name)

    reg_raw = _first_match(r"Beacon Registration Data\s+Available:\s*(Yes|No)", compact, re.I) or _first_match(r"Available:\s*(Yes|No)", compact, re.I)
    valid_reg = None if not reg_raw else reg_raw.lower() == "yes"
    vehicle_type, vehicle_name = _parse_vehicle_form(raw_no_header)
    flag_code, flag_country, call_sign = _parse_flag_and_call_form(raw_no_header)
    # More precise call-sign fallback
    if not call_sign:
        cs = _value_on_same_or_next(raw_no_header, r"Call Sign\s*:", max_lookahead=3)
        if cs and not re.fullmatch(r"(?:N/A|0|CAN|USA|AUS|NZL|FRA|CMB)", cs, re.I) and cs not in KNOWN_COUNTRIES_FOR_OLD:
            call_sign = cs
    if call_sign in {"0", "N/A", "Not Applicable", "Vehicle", "Vehicle type", "Vehicle name"}:
        call_sign = None

    # The 2014 form text often places labels first and the row values later:
    # `Cospas-Sarsat Provided: Persons Involved: Persons Rescued: first alert 2 0`.
    mp = re.search(r"Persons Involved:\s*(\d+(?:\.\d+)?)", compact, re.I)
    mr = re.search(r"Persons Rescued:\s*(\d+(?:\.\d+)?)", compact, re.I)
    assistance = _first_match(r"Cospas-Sarsat Provided:\s*(data not used in SAR|supporting data|first alert|only alert)", compact, re.I)
    row = re.search(r"Cospas-Sarsat Provided:\s*Persons Involved:\s*Persons Rescued:\s*(data not used in SAR|supporting data|first alert|only alert)\s+(\d+(?:\.\d+)?)\s+(\d+(?:\.\d+)?)", compact, re.I)
    if row:
        assistance = row.group(1)
        persons = to_float(row.group(2))
        saves = to_float(row.group(3))
    else:
        persons = to_float(mp.group(1)) if mp else None
        saves = to_float(mr.group(1)) if mr else None
    assistance = assistance.lower() if assistance else None

    details = None
    mdet = re.search(r"Details of Incident\s*(.*)", raw_no_header, re.I | re.S)
    if mdet:
        details = cleanup_record_text(mdet.group(1))
        # Remove any trailing next event header if present.
        details = re.split(r"\bEVENT\s*:\s*\d+\s+Reporting MCC", details, maxsplit=1, flags=re.I)[0].strip()
    if reporting_mcc == "CMCC" and not bc_code:
        bc_code, bc_name = "316", "Canada"
    if reporting_mcc == "CMCC" and not flag_country:
        flag_code, flag_country = "CAN", "Canada"
    vehicle_name = vehicle_name or "N/A"
    call_sign = call_sign or "N/A"
    flag_code = flag_code or "N/A"
    flag_country = flag_country or "N/A"

    return {
        "event_id": f"{report_year or 'unknown'}-{int(evnum or seq):04d}",
        "reporting_mcc": reporting_mcc,
        "incident_date": incident_date,
        "incident_date_raw": incident_date_raw,
        "alert_time_gmt": alert_time,
        "incident_type": incident_type,
        "location_description": location_description,
        "latitude_degrees": lat_d,
        "latitude_minutes": lat_m,
        "latitude_hemisphere": lat_h,
        "longitude_degrees": lon_d,
        "longitude_minutes": lon_m,
        "longitude_hemisphere": lon_h,
        "latitude_decimal": decimal_coord(lat_d, lat_m, lat_h),
        "longitude_decimal": decimal_coord(lon_d, lon_m, lon_h),
        "beacon_type": beacon_type.upper() if beacon_type else None,
        "beacon_hex": beacon_hex.upper() if beacon_hex and beacon_hex != "N/A" else None,
        "beacon_frequency_mhz": beacon_frequency_mhz,
        "beacon_country_code": bc_code,
        "beacon_country_name": bc_name,
        "valid_beacon_registration_available": valid_reg,
        "vehicle_type": vehicle_type,
        "vehicle_name": vehicle_name,
        "vehicle_flag_code": flag_code,
        "vehicle_flag_country": flag_country,
        "vehicle_call_sign": call_sign,
        "assistance_type": assistance,
        "number_persons_involved": persons,
        "cospas_sarsat_assisted_saves": saves,
        "incident_details": details,
        "source_pdf": source_pdf,
        "source_pdf_page": source_pdf_page,
        "report_year": report_year,
        "raw_record_text": cleanup_record_text(raw_no_header),
    }


def _parse_decimal_coord_from_table(chunk):
    mlat = re.search(r"(-?\d{1,3}(?:\.\d+)?)\s*°\s*([NS])", chunk, re.I)
    mlon = re.search(r"(-?\d{1,3}(?:\.\d+)?)\s*°\s*([EW])", chunk, re.I)
    lat_d = lat_m = lat_h = lon_d = lon_m = lon_h = lat_dec = lon_dec = None
    if mlat:
        val = abs(float(mlat.group(1)))
        lat_h = {"N":"North", "S":"South"}[mlat.group(2).upper()]
        lat_dec = val if lat_h == "North" else -val
        lat_d = int(val); lat_m = round((val-lat_d)*60, 6)
    if mlon:
        val = abs(float(mlon.group(1)))
        lon_h = {"E":"East", "W":"West"}[mlon.group(2).upper()]
        lon_dec = val if lon_h == "East" else -val
        lon_d = int(val); lon_m = round((val-lon_d)*60, 6)
    return lat_d, lat_m, lat_h, lon_d, lon_m, lon_h, lat_dec, lon_dec


def _date_from_table_start(raw_date, report_year):
    m = re.match(r"(\d{1,2})\s+([A-Za-z]+)", raw_date or "")
    if not m:
        return None
    d, mo = m.groups()
    return f"{int(report_year):04d}-{MONTH_NUM[mo.lower()]:02d}-{int(d):02d}" if report_year and mo.lower() in MONTH_NUM else None


def _extract_country_paren(chunk):
    # Country code in the table appears as `(Canada)` / `(USA)` / `(Australia)` etc.
    parens = re.findall(r"\(([^()]{2,60})\)", chunk)
    for p in reversed(parens):
        p = norm_ws(p)
        if p and not re.search(r"SRR|HH:MM|0-90|0-180", p, re.I):
            return p
    # fallback by country names in chunk
    for country in KNOWN_COUNTRIES_FOR_OLD:
        if re.search(rf"\b{re.escape(country)}\b", chunk, re.I):
            return country
    return None


def _parse_table_layout_record(chunk, source_pdf, source_pdf_page, report_year, seq):
    raw = _strip_form_headers(chunk)
    lines = _nonempty_lines(raw)
    if not lines:
        return None
    first = lines[0]
    mstart = re.match(r"\s*(\d{1,2}\s+[A-Za-z]+)\b", first)
    if not mstart:
        return None
    incident_date_raw = mstart.group(1)
    incident_date = _date_from_table_start(incident_date_raw, report_year)
    if not incident_date:
        return None

    compact = collapsed_record(raw)
    lat_d, lat_m, lat_h, lon_d, lon_m, lon_h, lat_dec, lon_dec = _parse_decimal_coord_from_table(raw)
    reporting_mcc = _first_match(r"\b([A-Z]{1,6}MCC)\b", compact, re.I)
    if reporting_mcc:
        reporting_mcc = reporting_mcc.upper()
    mp = re.search(r"\b[A-Z]{1,6}MCC\s+(\d+)\s*/?\s*(\d+)", compact, re.I)
    persons = to_float(mp.group(1)) if mp else None
    saves = to_float(mp.group(2)) if mp else None
    assistance = _first_match(r"Cospas-Sarsat provided\s+(data not used in SAR|supporting data|first alert|only alert)", compact, re.I)
    assistance = assistance.lower() if assistance else None
    frequency_raw = _first_match(r"(121\.5/243 MHz|406 MHz|406\.\d+ MHz)", compact, re.I)
    freq = None
    if frequency_raw:
        mf = re.search(r"(\d+(?:\.\d+)?)", frequency_raw)
        freq = float(mf.group(1)) if mf else None

    incident_type = None
    mtype = re.search(r"\b(Aviation|Maritime|Land)\b", compact, re.I)
    if mtype:
        incident_type = mtype.group(1).title()
    elif re.search(r"\bPLB\b", compact):
        incident_type = "Land"

    # Vehicle text: rows after coordinates and event type, before MCC. Use conservative hints.
    vehicle_type = None
    known_vehicle_words = ["Fishing vessel", "Sailing vessel", "Small vessel", "Motor vessel", "Vessel", "Boat", "Helicopter", "Plane", "Aircraft", "Person", "Tramper", "Camper", "Snowmobile", "Bushwalker", "Dinghy", "Yacht", "Catamaran"]
    for v in known_vehicle_words:
        if re.search(rf"\b{re.escape(v)}\b", compact, re.I):
            vehicle_type = v
            break
    if not vehicle_type:
        # pick line between coordinate lines and Details that is not country/mcc/frequency/assistance
        for ln in lines[:12]:
            if re.search(r"MCC|MHz|Cospas|Details|\d+\s+[A-Za-z]+|°|/|\(|\)", ln):
                continue
            if re.search(r"[A-Za-z]", ln) and len(ln.strip()) < 60:
                vehicle_type = ln.strip(); break

    vehicle_name = None
    call_sign = None
    # candidate lines before Details that aren't labels or countries. Often vehicle name/call sign are standalone under vehicle.
    pre_details = raw.split("Details", 1)[0]
    candidates = []
    for ln in _nonempty_lines(pre_details):
        s = ln.strip()
        if re.search(r"Date|Location|C/S Report|Page|Reporting|MCC|Beacon|No\. Persons|Vehicle|Country Code|Type of Event|Used|Cospas|Details|MHz|°|^/?$", s, re.I):
            continue
        if re.fullmatch(r"\(?[A-Za-z .'-]+\)?", s) and len(s) < 50:
            candidates.append(s.strip("()"))
        elif re.fullmatch(r"[A-Z0-9\-]{3,12}", s):
            call_sign = s
    country_name = _extract_country_paren(raw)
    for cand in candidates:
        if country_name and cand.lower() == country_name.lower():
            continue
        if cand in KNOWN_COUNTRIES_FOR_OLD:
            continue
        if vehicle_type and cand.lower() == vehicle_type.lower():
            continue
        # Avoid assistance/location fragments
        if re.search(r"alert|data|SAR|Cospas|provided", cand, re.I):
            continue
        vehicle_name = cand
        break

    # Location description: line after Details up to assistance phrase, or before `Cospas-Sarsat provided`.
    location_description = None
    if "Details" in raw:
        after_details = raw.split("Details", 1)[1]
        for ln in _nonempty_lines(after_details):
            if re.search(r"Cospas-Sarsat provided", ln, re.I):
                loc_part = re.split(r"Cospas-Sarsat provided", ln, flags=re.I)[0].strip()
                if loc_part:
                    location_description = loc_part
                    break
            if re.search(r"^[(/]?$", ln):
                continue
            if not re.search(r"Cospas|provided|^/$", ln, re.I):
                location_description = ln.strip()
                break
    if location_description:
        # Strip trailing assistance phrase if it was glued onto the same line.
        location_description = re.sub(r"\s+(data not used in SAR|supporting data|first alert|only alert)\s*$", "", location_description, flags=re.I).strip()

    details = None
    if assistance:
        mdet = re.search(r"Cospas-Sarsat provided\s+(?:data not used in SAR|supporting data|first alert|only alert)\s*(.*)", raw, re.I | re.S)
        if mdet:
            details = cleanup_record_text(mdet.group(1))
            details = re.split(r"\n?/?\s*$", details)[0].strip()
            details = re.sub(r"\s*/\s*$", "", details).strip()
    if not details and "Details" in raw:
        details = cleanup_record_text(raw.split("Details", 1)[1])

    beacon_country_name = country_name
    beacon_country_code = COUNTRY_CODE_BY_NAME.get(beacon_country_name) or (COUNTRY_CODE_MAP.get(beacon_country_name) if beacon_country_name else None)
    vehicle_flag_country = country_name
    vehicle_flag_code = FLAG_CODE_BY_COUNTRY.get(vehicle_flag_country)
    if reporting_mcc == "CMCC" and not beacon_country_code:
        beacon_country_code, beacon_country_name = "316", "Canada"
    if reporting_mcc == "CMCC" and not vehicle_flag_country:
        vehicle_flag_code, vehicle_flag_country = "CAN", "Canada"
    vehicle_name = vehicle_name or "N/A"
    call_sign = call_sign or "N/A"
    vehicle_type = vehicle_type or "N/A"
    vehicle_flag_code = vehicle_flag_code or "N/A"
    vehicle_flag_country = vehicle_flag_country or "N/A"
    # These table-layout files generally do not expose Beacon Hex ID; keep it null instead of fabricating.
    return {
        "event_id": f"{report_year or 'unknown'}-{seq:04d}",
        "reporting_mcc": reporting_mcc,
        "incident_date": incident_date,
        "incident_date_raw": incident_date_raw,
        "alert_time_gmt": None,
        "incident_type": incident_type,
        "location_description": clean_value_until_label(location_description),
        "latitude_degrees": lat_d,
        "latitude_minutes": lat_m,
        "latitude_hemisphere": lat_h,
        "longitude_degrees": lon_d,
        "longitude_minutes": lon_m,
        "longitude_hemisphere": lon_h,
        "latitude_decimal": lat_dec,
        "longitude_decimal": lon_dec,
        "beacon_type": None,
        "beacon_hex": None,
        "beacon_frequency_mhz": freq,
        "beacon_country_code": beacon_country_code,
        "beacon_country_name": beacon_country_name,
        "valid_beacon_registration_available": None,
        "vehicle_type": vehicle_type,
        "vehicle_name": vehicle_name,
        "vehicle_flag_code": vehicle_flag_code,
        "vehicle_flag_country": vehicle_flag_country,
        "vehicle_call_sign": call_sign,
        "assistance_type": assistance,
        "number_persons_involved": persons,
        "cospas_sarsat_assisted_saves": saves,
        "incident_details": details,
        "source_pdf": source_pdf,
        "source_pdf_page": source_pdf_page,
        "report_year": report_year,
        "raw_record_text": cleanup_record_text(raw),
    }


def _page_for_abs_pos(full, abs_pos):
    markers = re.findall(r"\[\[PDF_PAGE:(\d+)\]\]", full[:abs_pos])
    return int(markers[-1]) if markers else None


def _annex_region(full):
    starts = [m.start() for m in re.finditer(r"ANNEX\s+C|Annex\s+C|LIST OF SEARCH AND RESCUE EVENTS", full, re.I)]
    return min(starts) if starts else 0


def parse_pdf_records(pdf_path: Path) -> tuple[list[dict], list[dict]]:
    """Parse all Annex C records from a PDF. Supports modern, old form, 2014 EVENT form, and 2007-2009 table layouts."""
    text_path = extract_text_with_pages(pdf_path)
    raw_text = text_path.read_text(encoding="utf-8", errors="replace")
    report_year = infer_report_year(raw_text, pdf_path)
    pages = raw_text.split("\f")
    full = "".join(f"\n[[PDF_PAGE:{page_no}]]\n{page_text}" for page_no, page_text in enumerate(pages, start=1))
    annex_start = _annex_region(full)
    annex = full[annex_start:]

    # 2014-style form records use EVENT numbers instead of CA/AU Event IDs; parse by EVENT markers.
    event_matches = list(re.finditer(r"EVENT\s*:\s*\d+\s+Reporting MCC\s*:\s*[A-Z0-9]+", annex, re.I))
    if event_matches and re.search(r"Date of Incident\s*\(DD/MM/YYYY\)", annex, re.I):
        records, diag = [], []
        for idx, m in enumerate(event_matches):
            start = m.start()
            end = event_matches[idx + 1].start() if idx + 1 < len(event_matches) else len(annex)
            chunk = annex[start:end]
            source_pdf_page = _page_for_abs_pos(full, annex_start + start)
            rec = _parse_event_form_record(chunk, pdf_path.name, source_pdf_page, report_year, idx + 1)
            if rec:
                records.append(rec)
        if not records:
            raise RuntimeError(f"EVENT-form records were found but none parsed in {pdf_path.name}")
        return records, diag

    # 2007-2009 table layout starts rows with dates like `02 January` and has no Date-of-Incident label.
    # Begin at first row after the list header/table/map pages.
    table_matches = list(re.finditer(r"(?m)^\s*(\d{1,2}\s+(?:January|February|March|April|May|June|July|August|September|October|November|December))\b", annex, re.I))
    has_table_header = re.search(r"Date\s+Location", annex, re.I) and re.search(r"Cospas-Sarsat provided", annex, re.I)
    if table_matches and has_table_header and not re.search(OLD_DATE_LABEL_RE + r"\s*:", annex, re.I):
        records, diag = [], []
        for idx, m in enumerate(table_matches):
            start = m.start()
            end = table_matches[idx + 1].start() if idx + 1 < len(table_matches) else len(annex)
            chunk = annex[start:end]
            # skip table of statistics rows by requiring coordinates and Cospas text in the chunk
            if "°" not in chunk or "Cospas-Sarsat provided" not in chunk:
                continue
            source_pdf_page = _page_for_abs_pos(full, annex_start + start)
            rec = _parse_table_layout_record(chunk, pdf_path.name, source_pdf_page, report_year, len(records) + 1)
            if rec:
                records.append(rec)
        if not records:
            raise RuntimeError(f"Table-layout record starts were found but none parsed in {pdf_path.name}")
        return records, diag

    # Otherwise use the existing modern/older Date-of-Incident parser from v6.
    first_modern_candidates = [p for p in [full.find("Reporting MCC:"), full.find("Event ID:")] if p >= 0]
    first_old_candidates = [m.start() for m in re.finditer(OLD_DATE_LABEL_RE + r"\s*:\s*" + OLD_DATE_VALUE_RE, full, flags=re.I)]
    candidates = first_modern_candidates + first_old_candidates
    if not candidates:
        raise RuntimeError(f"No incident record starts found in {pdf_path.name}; looked for Reporting MCC, Event ID, Date of Incident, EVENT, and old table date rows")
    first_record = min(candidates)
    annex_positions = [m.start() for m in re.finditer(r"ANNEX\s+C|Annex\s+C|LIST OF SEARCH AND RESCUE EVENTS", full, flags=re.I)]
    annex_start2 = max([p for p in annex_positions if p <= first_record], default=first_record)
    annex2 = full[annex_start2:]
    starts = record_start_matches(annex2)
    if not starts:
        raise RuntimeError(f"No incident record starts found in {pdf_path.name}")
    records, page_end_instances = [], []
    for idx, (start, start_text, start_kind) in enumerate(starts):
        end = starts[idx + 1][0] if idx + 1 < len(starts) else len(annex2)
        chunk = annex2[start:end]
        source_pdf_page = _page_for_abs_pos(full, annex_start2 + start)
        if start_kind == "old_date":
            rec = parse_old_annex_record(chunk, source_pdf=pdf_path.name, source_pdf_page=source_pdf_page, report_year=report_year, seq=idx + 1)
        else:
            rec = parse_record(chunk, source_pdf=pdf_path.name, source_pdf_page=source_pdf_page, report_year=report_year)
        if rec:
            records.append(rec)
            if start_kind != "old_date" and "[[PDF_PAGE:" in chunk[:1200]:
                page_end_instances.append({
                    "source_pdf": pdf_path.name,
                    "report_year": report_year,
                    "event_id": rec["event_id"],
                    "reporting_mcc": rec["reporting_mcc"],
                    "source_pdf_page": source_pdf_page,
                    "raw_record_head": cleanup_record_text(chunk[:800]),
                })
    return records, page_end_instances


In [ ]:
# Download and parse all reports
all_records = []
all_page_end_instances = []
yearly_profile = []
parse_errors = []

for url in PDF_URLS:
    try:
        pdf_path = download_pdf(url)
        print(f"\nParsing {pdf_path.name} ({pdf_path.stat().st_size:,} bytes, md5={file_md5(pdf_path)})")
        records, page_end_instances = parse_pdf_records(pdf_path)
        canada_records = [r for r in records if is_canada_record(r)]

        report_year = records[0].get("report_year") if records else URL_YEAR_HINT.get(pdf_path.stem)
        yearly_profile.append({
            "report_year": report_year,
            "source_pdf": pdf_path.name,
            "source_url": url,
            "status": "parsed",
            "all_records": len(records),
            "unique_event_ids": len({r.get("event_id") for r in records}),
            "canada_records": len(canada_records),
            "canada_date_min": min([r.get("incident_date") for r in canada_records if r.get("incident_date")], default=None),
            "canada_date_max": max([r.get("incident_date") for r in canada_records if r.get("incident_date")], default=None),
            "page_end_event_id_instances": len(page_end_instances),
        })
        all_records.extend(records)
        all_page_end_instances.extend(page_end_instances)
    except Exception as exc:
        error_row = {
            "source_url": url,
            "source_pdf": url.rstrip('/').split('/')[-1],
            "report_year": URL_YEAR_HINT.get(Path(url.rstrip('/').split('/')[-1]).stem),
            "status": "error",
            "error": repr(exc),
        }
        yearly_profile.append(error_row)
        parse_errors.append(error_row)
        print(f"ERROR processing {url}: {exc!r}")
        if STOP_ON_DOWNLOAD_ERROR:
            raise

print(f"Parsed {len(all_records):,} records from {sum(1 for r in yearly_profile if r.get('status') == 'parsed')} PDFs")
if parse_errors:
    print(f"Encountered {len(parse_errors)} PDF errors. See outputs/cospas_sarsat_parse_errors.csv after the output cell runs.")


In [ ]:
# Filter Canada-only records and write output files.
canada_all_years = [r for r in all_records if is_canada_record(r)]
canada_all_years.sort(key=lambda r: (r.get("incident_date") or "", r.get("report_year") or 0, r.get("event_id") or ""))

json_path = OUTPUT_DIR / "cospas_sarsat_incident_records_canada_2007_2024.json"
csv_path = OUTPUT_DIR / "cospas_sarsat_incident_records_canada_2007_2024.csv"
profile_path = OUTPUT_DIR / "cospas_sarsat_yearly_profile.json"
page_end_path = OUTPUT_DIR / "cospas_sarsat_page_end_event_id_instances.csv"
parse_errors_path = OUTPUT_DIR / "cospas_sarsat_parse_errors.csv"

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(canada_all_years, f, ensure_ascii=False, indent=2)

pd.DataFrame(canada_all_years).to_csv(csv_path, index=False, encoding="utf-8")
with open(profile_path, "w", encoding="utf-8") as f:
    json.dump(yearly_profile, f, ensure_ascii=False, indent=2)

pd.DataFrame(all_page_end_instances).to_csv(page_end_path, index=False, encoding="utf-8")
pd.DataFrame(parse_errors).to_csv(parse_errors_path, index=False, encoding="utf-8")

print(f"Wrote {len(canada_all_years):,} Canada-only records")
print(json_path)
print(csv_path)
print(profile_path)
print(page_end_path)
print(parse_errors_path)

pd.DataFrame(yearly_profile).sort_values(["report_year", "source_pdf"], na_position="first")


In [ ]:
# Basic validation checks
canada_df = pd.DataFrame(canada_all_years)

summary = {
    "total_canada_records": len(canada_all_years),
    "unique_event_ids_with_year": len({(r.get("report_year"), r.get("event_id")) for r in canada_all_years}),
    "date_min": canada_df["incident_date"].dropna().min() if not canada_df.empty and "incident_date" in canada_df.columns else None,
    "date_max": canada_df["incident_date"].dropna().max() if not canada_df.empty and "incident_date" in canada_df.columns else None,
    "records_by_year": canada_df.groupby("report_year")["event_id"].count().to_dict() if not canada_df.empty else {},
    "incident_type_counts": canada_df["incident_type"].value_counts(dropna=False).to_dict() if not canada_df.empty else {},
}
summary

In [ ]:
# Target-field completeness check for the fields that are most sensitive to PDF layout parsing.
TARGET_FIELDS = [
    "beacon_country_code", "beacon_country_name", "vehicle_type", "vehicle_name",
    "vehicle_flag_code", "vehicle_flag_country", "vehicle_call_sign",
    "assistance_type", "number_persons_involved", "cospas_sarsat_assisted_saves",
]

def is_empty(v):
    return v is None or (isinstance(v, float) and pd.isna(v)) or (isinstance(v, str) and v.strip() == "")

quality_rows = []
for field in TARGET_FIELDS:
    empty_count = int(canada_df[field].map(is_empty).sum()) if field in canada_df.columns else len(canada_df)
    quality_rows.append({
        "field": field,
        "non_empty": len(canada_df) - empty_count,
        "empty": empty_count,
        "empty_pct": round(empty_count / len(canada_df) * 100, 2) if len(canada_df) else 0,
        "unique_non_empty": int(canada_df.loc[~canada_df[field].map(is_empty), field].astype(str).nunique()) if field in canada_df.columns else 0,
    })

quality_df = pd.DataFrame(quality_rows)
display(quality_df)

# Show rows needing review, if any.
review_mask = pd.Series(False, index=canada_df.index)
for field in TARGET_FIELDS:
    if field in canada_df.columns:
        review_mask = review_mask | canada_df[field].map(is_empty)

review_cols = ["report_year", "event_id", "source_pdf", "source_pdf_page", *TARGET_FIELDS, "raw_record_text"]
rows_needing_review = canada_df.loc[review_mask, [c for c in review_cols if c in canada_df.columns]]
print(f"Rows needing target-field review: {len(rows_needing_review):,}")
display(rows_needing_review.head(50))

In [ ]:
# Optional: inspect the known page-break issue pattern from 2022.
# CA-023 should have its event header at the bottom of the previous page and the body below it.
if not canada_df.empty and "event_id" in canada_df.columns:
    display(canada_df.loc[canada_df["event_id"].eq("CA-023"), [
        "event_id", "report_year", "incident_date", "location_description", "source_pdf_page", "raw_record_text"
    ]].head(3))